# PM2.5 spatio-temporal — Kaggle notebook from current `code/`

This notebook was built from scratch from the current scripts in the `code/` directory.
Kaggle setup -> project file preparation -> pipeline execution -> artifact collection.

**Kaggle setup:** language **R**, **GPU** accelerator for full retraining, **Internet ON**.
The notebook materializes the current `code/*.R` files into `/kaggle/working/code/` and then runs them in the project’s natural order.


## 1. Configuration

Fill in your ADS/CAMS credentials below. The notebook intentionally uses placeholders instead of hardcoded secrets.


In [ ]:
CDS_USER <- ""
CDS_API_KEY <- ""

Sys.setenv(TRAINING_PROFILE = "gpu")
options(timeout = 3600)
say <- function(...) cat(sprintf("\n========== %s ==========\n", sprintf(...)))

if (!nzchar(CDS_USER) || !nzchar(CDS_API_KEY)) {
  message("Fill CDS_USER and CDS_API_KEY before running the data-acquisition step.")
}


## 2. Kaggle working directory and project structure


In [ ]:
setwd("/kaggle/working")
for (d in c("code", "code_kaggle", "data/raw", "data/raw/spatial", "data/processed",
            "output/figures", "output/tables", "models", "docs")) {
  dir.create(d, recursive = TRUE, showWarnings = FALSE)
}
writeLines("", ".here")
if (nzchar(CDS_USER) && nzchar(CDS_API_KEY)) {
  writeLines(c(paste0("CDS_USER=", CDS_USER), paste0("CDS_API_KEY=", CDS_API_KEY)), ".env")
} else {
  writeLines(c("CDS_USER=", "CDS_API_KEY="), ".env")
}
cat("cwd:", getwd(), "| project directories ready\n")


## 3. System libraries for `sf`, `terra`, `stars`, and NetCDF


In [ ]:
say("install apt libraries")
cmd <- paste(
  "apt-get update -qq &&",
  "apt-get install -y -qq libgdal-dev libgeos-dev libproj-dev libnetcdf-dev libudunits2-dev"
)
try(system(cmd), silent = TRUE)


## 4. Materialize the current `code/` files

This cell writes the current project scripts into `/kaggle/working/code/`.


In [ ]:
write_code_file <- function(path, lines) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  writeLines(lines, path, useBytes = TRUE)
  cat(sprintf("wrote %s (%d lines)\n", path, length(lines)))
}

files_written <- character()

write_code_file("code/_common_training.R", c(
  "suppressPackageStartupMessages({",
  "  library(torch)",
  "  library(luz)",
  "  library(tidyverse)",
  "  library(here)",
  "})",
  "",
  "set.seed(42)",
  "torch::torch_manual_seed(42)",
  "",
  "# Training profile, selectable at the shell with e.g.",
  "#   TRAINING_PROFILE=gpu Rscript code/run_all_after_data_acquisition.R",
  "#   \"cpu\" (default): every architecture trains at batch 4, lr 1e-3, on CPU.",
  "#   \"gpu\": per-cell LSTM stays at batch 4 / lr 1e-3; CNN-LSTM and ConvLSTM use",
  "#          batch 64 with lr scaled by the square-root rule (lr = 1e-3 * sqrt(batch/4)),",
  "#          on CUDA when available (falls back to CPU otherwise; Apple MPS is",
  "#          intentionally avoided — bilinear upsample issues on the 110x60 grid).",
  "# Only matters on a fresh retrain; inference via load_or_train() ignores it.",
  "TRAINING_PROFILE <- Sys.getenv(\"TRAINING_PROFILE\", \"cpu\") # \"cpu\" | \"gpu\"",
  "",
  "LR_BASE_BATCH <- 4L",
  "LR_BASE <- 1e-3",
  "lr_for_batch <- function(batch) LR_BASE * sqrt(batch / LR_BASE_BATCH)",
  "",
  "EPOCHS <- 100L",
  "PATIENCE <- 20L",
  "STEP_SIZE <- 15L",
  "GAMMA <- 0.5",
  "",
  ".PROFILE_BATCH <- list(",
  "  cpu = list(percell = 4L, spatial = 4L),",
  "  gpu = list(percell = 4L, spatial = 64L)",
  ")",
  "",
  "# Resolve (batch_size, lr, accelerator) for an architecture family",
  "# (\"spatial\" = CNN-LSTM & ConvLSTM, \"percell\" = per-cell LSTM) under the",
  "# active profile. Each model script calls this once.",
  "arch_hparams <- function(arch = c(\"percell\", \"spatial\")) {",
  "  arch <- match.arg(arch)",
  "  bs <- .PROFILE_BATCH[[TRAINING_PROFILE]][[arch]]",
  "  list(",
  "    batch_size = bs,",
  "    lr = lr_for_batch(bs),",
  "    accelerator = if (identical(TRAINING_PROFILE, \"gpu\") && torch::cuda_is_available()) {",
  "      luz::accelerator()",
  "    } else {",
  "      luz::accelerator(cpu = TRUE)",
  "    }",
  "  )",
  "}",
  "",
  "BATCH_SIZE <- .PROFILE_BATCH[[TRAINING_PROFILE]]$percell # 4 under cpu",
  "LR <- lr_for_batch(BATCH_SIZE) # 1e-3 under cpu",
  "",
  "cpu_acc <- luz::accelerator(cpu = TRUE)",
  "",
  "pm25_dataset <- torch::dataset(",
  "  name = \"pm25_dataset\",",
  "  initialize = function(source_array, Y, sample_indices, window_size) {",
  "    self$src <- source_array",
  "    self$Y <- Y",
  "    self$idx <- sample_indices",
  "    self$win <- window_size",
  "  },",
  "  .getitem = function(i) {",
  "    t_start <- self$idx[i]",
  "    t_end <- t_start + self$win - 1",
  "    x <- torch::torch_tensor(self$src[, , t_start:t_end, ])$permute(c(3, 4, 1, 2))",
  "    y <- torch::torch_tensor(self$Y[i, , ])$unsqueeze(1)",
  "    list(x = x, y = y)",
  "  },",
  "  .length = function() nrow(self$Y)",
  ")",
  "",
  "make_dataloaders <- function(tensors, meta, batch_size = BATCH_SIZE) {",
  "  src <- tensors$source_array",
  "  list(",
  "    train = torch::dataloader(",
  "      pm25_dataset(src, tensors$Y_train, meta$split_idx$train, meta$window_size),",
  "      batch_size = batch_size, shuffle = TRUE",
  "    ),",
  "    val = torch::dataloader(",
  "      pm25_dataset(src, tensors$Y_val, meta$split_idx$val, meta$window_size),",
  "      batch_size = batch_size, shuffle = FALSE",
  "    ),",
  "    test = torch::dataloader(",
  "      pm25_dataset(src, tensors$Y_test, meta$split_idx$test, meta$window_size),",
  "      batch_size = batch_size, shuffle = FALSE",
  "    )",
  "  )",
  "}",
  "",
  "make_callbacks <- function(log_path) {",
  "  list(",
  "    luz::luz_callback_early_stopping(",
  "      monitor = \"valid_loss\",",
  "      patience = PATIENCE,",
  "      mode = \"min\"",
  "    ),",
  "    luz::luz_callback_keep_best_model(",
  "      monitor = \"valid_loss\",",
  "      mode = \"min\"",
  "    ),",
  "    luz::luz_callback_lr_scheduler(",
  "      lr_scheduler = torch::lr_step,",
  "      step_size = STEP_SIZE,",
  "      gamma = GAMMA",
  "    ),",
  "    luz::luz_callback_gradient_clip(max_norm = 1.0),",
  "    luz::luz_callback_csv_logger(path = log_path)",
  "  )",
  "}",
  "",
  "make_inv_scale <- function(meta) {",
  "  function(x) {",
  "    unscaled <- x * (meta$pm25_max - meta$pm25_min) + meta$pm25_min",
  "    if (isTRUE(meta$log_transform)) expm1(unscaled) else unscaled",
  "  }",
  "}",
  "",
  "persistence_pm25 <- function(tensors, meta, inv_scale = make_inv_scale(meta)) {",
  "  src <- tensors$source_array",
  "  n_test <- length(meta$split_idx$test)",
  "  win <- meta$window_size",
  "  out <- array(dim = c(n_test, dim(src)[1], dim(src)[2]))",
  "  for (i in seq_len(n_test)) {",
  "    t_last <- meta$split_idx$test[i] + win - 1",
  "    out[i, , ] <- src[, , t_last, 1]",
  "  }",
  "  inv_scale(out)",
  "}",
  "",
  "fit_shared <- function(module, hparams, dls, log_path,",
  "                       lr = LR, accelerator = cpu_acc) {",
  "  stage1 <- luz::setup(module,",
  "    loss      = torch::nn_mse_loss(),",
  "    optimizer = torch::optim_adam,",
  "    metrics   = list(luz::luz_metric_mae())",
  "  )",
  "  stage2 <- do.call(luz::set_hparams, c(list(stage1), hparams))",
  "  stage3 <- luz::set_opt_hparams(stage2, lr = lr)",
  "  luz::fit(stage3,",
  "    data        = dls$train,",
  "    valid_data  = dls$val,",
  "    epochs      = EPOCHS,",
  "    accelerator = accelerator,",
  "    callbacks   = make_callbacks(log_path),",
  "    verbose     = TRUE",
  "  )",
  "}",
  "",
  "evaluate_and_save <- function(fitted, test_dl, tensors, meta,",
  "                              results_path, label,",
  "                              accelerator = cpu_acc) {",
  "  inv_scale <- make_inv_scale(meta)",
  "  test_actual_pm25 <- inv_scale(tensors$Y_test)",
  "",
  "  preds <- predict(fitted, test_dl, accelerator = accelerator)",
  "  preds_pm <- pmax(inv_scale(as.array(preds$cpu())[, 1, , ]), 0)",
  "",
  "  rmse <- sqrt(mean((preds_pm - test_actual_pm25)^2, na.rm = TRUE))",
  "  mae <- mean(abs(preds_pm - test_actual_pm25), na.rm = TRUE)",
  "  mae_grid <- apply(abs(preds_pm - test_actual_pm25), c(2, 3), mean, na.rm = TRUE)",
  "",
  "  cat(sprintf(\"\\n%s: RMSE = %.2f µg/m³, MAE = %.2f\\n\", label, rmse, mae))",
  "",
  "  result <- list(",
  "    predictions = preds_pm, actuals = test_actual_pm25,",
  "    rmse = rmse, mae = mae, mae_grid = mae_grid",
  "  )",
  "  saveRDS(result, results_path)",
  "  cat(sprintf(\"Saved: %s\\n\", results_path))",
  "",
  "  invisible(result)",
  "}",
  "",
  "save_training_curve <- function(log_path, fig_path, title,",
  "                                loss_label = \"MSE\") {",
  "  tl <- readr::read_csv(log_path, show_col_types = FALSE)",
  "  p <- ggplot2::ggplot(tl, ggplot2::aes(epoch, loss, color = set)) +",
  "    ggplot2::geom_line(linewidth = 1) +",
  "    ggplot2::geom_point(size = 1.2) +",
  "    ggplot2::labs(title = title, y = loss_label, x = \"Epoch\", color = NULL) +",
  "    ggplot2::scale_color_manual(",
  "      values = c(train = \"steelblue\", valid = \"tomato\"),",
  "      labels = c(train = \"Train\", valid = \"Validation\")",
  "    ) +",
  "    ggplot2::theme_minimal()",
  "  out_dir <- dirname(fig_path)",
  "  if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)",
  "  ggplot2::ggsave(fig_path, p, width = 8, height = 5)",
  "}",
  "",
  "MODEL_DIR <- here::here(\"models\")",
  "",
  "load_or_train <- function(module, hparams, dls, log_path, model_filename,",
  "                          lr = LR, accelerator = cpu_acc,",
  "                          fig_path = NULL, fig_title = NULL) {",
  "  path <- file.path(MODEL_DIR, model_filename)",
  "  if (file.exists(path)) {",
  "    cat(sprintf(\"[load_or_train] loading pre-trained checkpoint: %s\\n\", path))",
  "    cat(\"[load_or_train] skipping training (delete the .pt to force a retrain)\\n\")",
  "    return(luz::luz_load(path))",
  "  }",
  "  cat(sprintf(\"[load_or_train] no checkpoint at %s -- training fresh\\n\", path))",
  "  fitted <- fit_shared(module, hparams, dls, log_path, lr = lr, accelerator = accelerator)",
  "  if (!dir.exists(MODEL_DIR)) dir.create(MODEL_DIR, recursive = TRUE)",
  "  luz::luz_save(fitted, path)",
  "  cat(sprintf(\"[load_or_train] saved freshly trained model: %s\\n\", path))",
  "  if (!is.null(fig_path) && !is.null(fig_title)) {",
  "    save_training_curve(log_path, fig_path, fig_title)",
  "  }",
  "  fitted",
  "}"
))
files_written <- c(files_written, "code/_common_training.R")

write_code_file("code/00_install.R", c(
  "packages <- c(",
  "  \"ecmwfr\",",
  "  \"dotenv\",",
  "",
  "  \"stars\",",
  "  \"sf\",",
  "  \"spdep\",",
  "  \"terra\",",
  "  \"elevatr\",",
  "  \"rnaturalearth\",",
  "  \"rnaturalearthdata\",",
  "",
  "  \"torch\",",
  "  \"luz\",",
  "",
  "  \"tidyverse\",",
  "  \"here\",",
  "  \"cowplot\",",
  "  \"patchwork\",",
  "  \"plotly\",",
  "  \"quarto\",",
  "  \"rmarkdown\"",
  ")",
  "",
  "installed <- rownames(installed.packages())",
  "to_install <- setdiff(packages, installed)",
  "",
  "if (length(to_install) > 0) {",
  "  message(sprintf(\"Installing %d packages: %s\", length(to_install),",
  "                  paste(to_install, collapse = \", \")))",
  "  install.packages(to_install, repos = \"https://cloud.r-project.org\")",
  "} else {",
  "  message(\"All packages already installed.\")",
  "}",
  "",
  "if (requireNamespace(\"torch\", quietly = TRUE)) {",
  "  if (!torch::torch_is_installed()) {",
  "    message(\"Installing LibTorch backend...\")",
  "    torch::install_torch()",
  "  } else {",
  "    message(\"LibTorch backend already installed.\")",
  "  }",
  "}",
  "",
  "message(\"Done. You can now run scripts 01 through 12.\")"
))
files_written <- c(files_written, "code/00_install.R")

write_code_file("code/01_data_acquisition.R", c(
  "suppressPackageStartupMessages({",
  "  library(ecmwfr)",
  "  library(terra)",
  "  library(sf)",
  "  library(here)",
  "})",
  "",
  "OUT_DIR <- here::here(\"data\", \"raw\")",
  "if (!dir.exists(OUT_DIR)) dir.create(OUT_DIR, recursive = TRUE)",
  "",
  "cams_expected <- sprintf(\"cams_pm25_poland_%d_%02d\",",
  "                         rep(2018:2022, each = 12), rep(1:12, times = 5))",
  "have_nc  <- all(file.exists(file.path(OUT_DIR, paste0(cams_expected, \".nc\"))))",
  "have_zip <- all(file.exists(file.path(OUT_DIR, paste0(cams_expected, \".zip\"))))",
  "",
  "if (have_nc || have_zip) {",
  "  message(sprintf(",
  "    \"All 60 CAMS monthly files already present under %s — skipping download.\",",
  "    OUT_DIR))",
  "} else {",
  "  dotenv::load_dot_env(here::here(\".env\"))",
  "  CDS_USER    <- Sys.getenv(\"CDS_USER\")",
  "  CDS_API_KEY <- Sys.getenv(\"CDS_API_KEY\")",
  "",
  "  if (CDS_USER == \"\" || CDS_API_KEY == \"\") {",
  "    stop(\"Missing CDS_USER or CDS_API_KEY in .env file. See .env.example\")",
  "  }",
  "  ecmwfr::wf_set_key(key = CDS_API_KEY, user = CDS_USER)",
  "",
  "  NORTH <- 55; WEST <- 14; SOUTH <- 49; EAST <- 25",
  "",
  "  YEARS  <- 2018:2022",
  "  MONTHS <- sprintf(\"%02d\", 1:12)",
  "",
  "  for (yr in YEARS) {",
  "    for (mo in MONTHS) {",
  "      target_file <- sprintf(\"cams_pm25_poland_%d_%s.nc\", yr, mo)",
  "      out_path    <- file.path(OUT_DIR, target_file)",
  "",
  "      if (file.exists(out_path)) {",
  "        message(sprintf(\"Skipping %s (already exists)\", target_file))",
  "        next",
  "      }",
  "",
  "      message(sprintf(\"Requesting PM2.5 for %d-%s ...\", yr, mo))",
  "",
  "      request <- list(",
  "        dataset_short_name = \"cams-europe-air-quality-reanalyses\",",
  "        variable    = \"particulate_matter_2.5um\",",
  "        model       = \"ensemble\",",
  "        level       = \"0\",",
  "        type        = \"validated_reanalysis\",",
  "        year        = as.character(yr),",
  "        month       = mo,",
  "        data_format = \"netcdf\",",
  "        area        = c(NORTH, WEST, SOUTH, EAST),",
  "        target      = target_file",
  "      )",
  "",
  "      tryCatch(",
  "        {",
  "          ecmwfr::wf_request(",
  "            request  = request,",
  "            transfer = TRUE,",
  "            path     = OUT_DIR,",
  "            user     = CDS_USER,",
  "            verbose  = TRUE",
  "          )",
  "          message(sprintf(\"  -> Saved: %s\", out_path))",
  "        },",
  "        error = function(e) {",
  "          message(sprintf(\"  -> ERROR for %d-%s: %s\", yr, mo, e$message))",
  "        }",
  "      )",
  "    }",
  "  }",
  "",
  "  message(\"CAMS download complete. Check data/raw/ for NetCDF files.\")",
  "}",
  "",
  "SPATIAL_OUT <- here::here(\"data\", \"processed\", \"spatial_features.rds\")",
  "",
  "GRID_NX   <- 110L",
  "GRID_NY   <- 60L",
  "LON_MIN   <- 14.0; LON_MAX <- 24.9; LON_STEP <-  0.1",
  "LAT_MAX   <- 55.0; LAT_MIN <- 49.1; LAT_STEP <- -0.1",
  "",
  "lon_centres <- LON_MIN + seq_len(GRID_NX) * LON_STEP - LON_STEP / 2",
  "lat_centres <- LAT_MAX + seq_len(GRID_NY) * LAT_STEP - LAT_STEP / 2",
  "",
  "template <- terra::rast(",
  "  xmin = LON_MIN - LON_STEP / 2, xmax = LON_MAX + LON_STEP / 2,",
  "  ymin = LAT_MIN + LAT_STEP / 2, ymax = LAT_MAX - LAT_STEP / 2,",
  "  ncols = GRID_NX, nrows = GRID_NY,",
  "  crs  = \"EPSG:4326\"",
  ")",
  "",
  "spatial_needs_build <- TRUE",
  "if (file.exists(SPATIAL_OUT)) {",
  "  existing <- readRDS(SPATIAL_OUT)",
  "  ok_shape <- identical(dim(existing$elevation), c(GRID_NX, GRID_NY)) &&",
  "              identical(dim(existing$log_pop_density), c(GRID_NX, GRID_NY))",
  "  if (ok_shape) {",
  "    message(sprintf(\"spatial_features.rds exists with correct shape — nothing to do. Delete %s to rebuild.\",",
  "                    SPATIAL_OUT))",
  "    spatial_needs_build <- FALSE",
  "  } else {",
  "    message(\"Existing spatial_features.rds has wrong shape; rebuilding.\")",
  "  }",
  "}",
  "",
  "if (spatial_needs_build) {",
  "  raw_spatial_dir <- here::here(\"data\", \"raw\", \"spatial\")",
  "  dir.create(raw_spatial_dir, recursive = TRUE, showWarnings = FALSE)",
  "",
  "  message(\"Fetching AWS Terrain Tiles DEM via elevatr (src='aws')…\")",
  "",
  "  bbox_sf <- sf::st_as_sfc(sf::st_bbox(c(",
  "    xmin = LON_MIN - 0.2, xmax = LON_MAX + 0.2,",
  "    ymin = LAT_MIN - 0.2, ymax = LAT_MAX + 0.2",
  "  ), crs = 4326))",
  "  bbox_sf <- sf::st_sf(geometry = bbox_sf, id = 1L)",
  "",
  "  dem_raster <- elevatr::get_elev_raster(",
  "    locations = bbox_sf,",
  "    z         = 6,",
  "    prj       = \"EPSG:4326\",",
  "    clip      = \"bbox\"",
  "  )",
  "  dem_terra <- terra::rast(dem_raster)",
  "  message(sprintf(\"  native DEM: %d x %d cells, res %.5f°\",",
  "                  terra::ncol(dem_terra), terra::nrow(dem_terra), terra::res(dem_terra)[1]))",
  "",
  "  elev_10km <- terra::resample(dem_terra, template, method = \"average\")",
  "",
  "  elev_mat <- t(matrix(as.vector(elev_10km), nrow = GRID_NY, ncol = GRID_NX,",
  "                       byrow = TRUE))",
  "  stopifnot(dim(elev_mat) == c(GRID_NX, GRID_NY))",
  "",
  "  elev_mat <- pmax(elev_mat, 0)",
  "",
  "  message(sprintf(\"  aggregated elevation: mean %.1f m, range [%.0f, %.0f] m, NAs: %d\",",
  "                  mean(elev_mat, na.rm = TRUE),",
  "                  min(elev_mat,  na.rm = TRUE),",
  "                  max(elev_mat,  na.rm = TRUE),",
  "                  sum(is.na(elev_mat))))",
  "",
  "  stopifnot(",
  "    \"Elevation grid has NAs — AWS DEM tile did not cover the full CAMS extent\" =",
  "      !anyNA(elev_mat),",
  "    \"Elevation range outside plausible bounds for Poland\" =",
  "      all(elev_mat >= 0 & elev_mat <= 2600)",
  "  )",
  "",
  "  elev_min <- min(elev_mat); elev_max <- max(elev_mat)",
  "  elevation_scaled <- (elev_mat - elev_min) / (elev_max - elev_min)",
  "",
  "  pop_url <- \"https://data.worldpop.org/GIS/Population/Global_2000_2020/2018/POL/pol_ppp_2018.tif\"",
  "  pop_tif <- file.path(raw_spatial_dir, \"pol_ppp_2018.tif\")",
  "",
  "  if (!file.exists(pop_tif)) {",
  "    message(\"Downloading WorldPop Poland 2018 TIF (~260 MB, a few minutes on typical broadband)…\")",
  "    old_timeout <- getOption(\"timeout\"); on.exit(options(timeout = old_timeout), add = TRUE)",
  "    options(timeout = 1800)",
  "    utils::download.file(pop_url, pop_tif, mode = \"wb\", quiet = FALSE)",
  "  }",
  "  stopifnot(\"WorldPop TIF did not download\" = file.exists(pop_tif))",
  "",
  "  pop_raster <- terra::rast(pop_tif)",
  "  message(sprintf(\"  native WorldPop: %d x %d cells, res %.5f°\",",
  "                  terra::ncol(pop_raster), terra::nrow(pop_raster), terra::res(pop_raster)[1]))",
  "",
  "  pop_sum <- terra::resample(pop_raster, template, method = \"sum\")",
  "",
  "  earth_r   <- 6371.0088",
  "  lat_rad   <- lat_centres * pi / 180",
  "  cell_area <- (LON_STEP * pi / 180) * (-LAT_STEP * pi / 180) *",
  "               earth_r^2 * cos(lat_rad)",
  "  cell_area_mat <- matrix(cell_area, nrow = GRID_NX, ncol = GRID_NY, byrow = TRUE)",
  "",
  "  pop_mat <- t(matrix(as.vector(pop_sum), nrow = GRID_NY, ncol = GRID_NX,",
  "                      byrow = TRUE))",
  "  stopifnot(dim(pop_mat) == c(GRID_NX, GRID_NY))",
  "",
  "  pop_mat[is.na(pop_mat)] <- 0",
  "",
  "  pop_density <- pop_mat / cell_area_mat",
  "  log_pop     <- log1p(pop_density)",
  "",
  "  message(sprintf(\"  aggregated pop density: mean %.1f /km², max %.1f /km², cells = 0: %d\",",
  "                  mean(pop_density), max(pop_density),",
  "                  sum(pop_density == 0)))",
  "",
  "  stopifnot(",
  "    \"Population density outside plausible bounds\" =",
  "      all(pop_density >= 0 & pop_density <= 30000)",
  "  )",
  "",
  "  log_pop_min <- min(log_pop); log_pop_max <- max(log_pop)",
  "  log_pop_scaled <- (log_pop - log_pop_min) / (log_pop_max - log_pop_min)",
  "",
  "  out <- list(",
  "    elevation        = elevation_scaled,",
  "    log_pop_density  = log_pop_scaled,",
  "    meta = list(",
  "      grid_dim          = c(nx = GRID_NX, ny = GRID_NY),",
  "      elev_min_m        = elev_min,",
  "      elev_max_m        = elev_max,",
  "      log_pop_min       = log_pop_min,",
  "      log_pop_max       = log_pop_max,",
  "      pop_density_max   = max(pop_density),",
  "      built_at          = Sys.time(),",
  "      elevation_source  = \"AWS Terrain Tiles via elevatr (src='aws'; composite of 3DEP, SRTM, GMTED2010, ETOPO1), z=6\",",
  "      population_source = \"WorldPop Global 2000-2020 Poland 2018 top-down unconstrained (~100 m / 3 arc-second native)\"",
  "    )",
  "  )",
  "",
  "  proc_dir <- here::here(\"data\", \"processed\")",
  "  if (!dir.exists(proc_dir)) dir.create(proc_dir, recursive = TRUE)",
  "  saveRDS(out, SPATIAL_OUT)",
  "",
  "  message(sprintf(\"Saved %s (%.1f KB)\", SPATIAL_OUT,",
  "                  file.info(SPATIAL_OUT)$size / 1024))",
  "}",
  "",
  "message(\"Data acquisition complete.\")"
))
files_written <- c(files_written, "code/01_data_acquisition.R")

write_code_file("code/02_data_prep.R", c(
  "library(stars)",
  "library(sf)",
  "library(spdep)",
  "library(rnaturalearth)",
  "library(rnaturalearthdata)",
  "library(tidyverse)",
  "library(here)",
  "",
  "raw_dir <- here::here(\"data\", \"raw\")",
  "",
  "zip_files <- sort(list.files(raw_dir,",
  "  pattern = \"cams_pm25_poland_.*\\\\.zip$\",",
  "  full.names = TRUE",
  "))",
  "if (length(zip_files) > 0) {",
  "  message(sprintf(\"Found %d zip files, extracting...\", length(zip_files)))",
  "  for (zf in zip_files) {",
  "    contents <- utils::unzip(zf, list = TRUE)$Name",
  "    missing <- contents[!file.exists(file.path(raw_dir, contents))]",
  "    if (length(missing) > 0) {",
  "      utils::unzip(zf, files = missing, exdir = raw_dir, overwrite = FALSE)",
  "    }",
  "  }",
  "}",
  "",
  "nc_files <- sort(list.files(raw_dir, pattern = \".*\\\\.nc$\", full.names = TRUE))",
  "",
  "if (length(nc_files) == 0) {",
  "  stop(\"No NetCDF files found in data/raw/. Run 01_data_acquisition.R first.\")",
  "}",
  "",
  "message(sprintf(\"Found %d NetCDF files, loading...\", length(nc_files)))",
  "",
  "message(\"Reading files and detecting grid dimensions...\")",
  "pm25_list <- lapply(nc_files, stars::read_stars)",
  "",
  "dims_y <- sapply(pm25_list, function(s) dim(s)[2])",
  "if (length(unique(dims_y)) > 1) {",
  "  message(sprintf(",
  "    \"  Grid dimension mismatch detected (y: %s). Trimming to common dimensions (registrations differ by half a cell between the 2018-19 and 2020-22 streams).\",",
  "    paste(unique(dims_y), collapse = \", \")",
  "  ))",
  "  min_ny <- min(dims_y)",
  "  min_nx <- min(sapply(pm25_list, function(s) dim(s)[1]))",
  "  pm25_list <- lapply(pm25_list, function(s) {",
  "    s[, seq_len(min_nx), seq_len(min_ny), ]",
  "  })",
  "  message(sprintf(\"  Harmonized all grids to %d x %d\", min_nx, min_ny))",
  "}",
  "",
  "message(\"Aggregating hourly -> daily per month...\")",
  "",
  "daily_arrays <- list()",
  "daily_dates <- c()",
  "",
  "for (i in seq_along(pm25_list)) {",
  "  s <- pm25_list[[i]]",
  "  arr <- as.array(s[[1]])",
  "  t_vals <- stars::st_get_dimension_values(s, 3)",
  "  dates_h <- as.Date(t_vals)",
  "  unique_days <- unique(dates_h)",
  "",
  "  for (j in seq_along(unique_days)) {",
  "    d <- unique_days[j]",
  "    idx <- which(dates_h == d)",
  "    daily_slice <- apply(arr[, , idx, drop = FALSE], c(1, 2), mean, na.rm = TRUE)",
  "    daily_arrays <- c(daily_arrays, list(daily_slice))",
  "  }",
  "  daily_dates <- c(daily_dates, as.character(unique_days))",
  "",
  "  if (i %% 12 == 0) message(sprintf(\"  ... processed %d / %d files\", i, length(pm25_list)))",
  "}",
  "",
  "pm25_daily_arr <- array(",
  "  unlist(daily_arrays),",
  "  dim = c(dim(daily_arrays[[1]]), length(daily_arrays))",
  ")",
  "daily_dates <- as.Date(daily_dates, format = \"%Y-%m-%d\")",
  "",
  "message(sprintf(",
  "  \"Daily array: %s (x, y, days=%d)\",",
  "  paste(dim(pm25_daily_arr), collapse = \" x \"), length(daily_dates)",
  "))",
  "message(sprintf(\"Date range: %s to %s\", min(daily_dates), max(daily_dates)))",
  "",
  "template <- pm25_list[[1]][, , , 1]",
  "pm25_raw <- stars::st_as_stars(list(pm25 = pm25_daily_arr))",
  "stars::st_dimensions(pm25_raw)[[1]] <- stars::st_dimensions(template)[[1]]",
  "stars::st_dimensions(pm25_raw)[[2]] <- stars::st_dimensions(template)[[2]]",
  "pm25_raw <- stars::st_set_dimensions(pm25_raw, 3,",
  "  values = as.POSIXct(daily_dates),",
  "  names  = \"time\"",
  ")",
  "names(stars::st_dimensions(pm25_raw)) <- c(\"x\", \"y\", \"time\")",
  "sf::st_crs(pm25_raw) <- 4326",
  "",
  "print(pm25_raw)",
  "message(sprintf(\"Daily grid dimensions: %s\", paste(dim(pm25_raw), collapse = \" x \")))",
  "",
  "pm25_array_raw <- pm25_daily_arr",
  "na_pct <- mean(is.na(pm25_array_raw)) * 100",
  "message(sprintf(\"Missing values: %.1f%%\", na_pct))",
  "",
  "na_mask_constant_raw <- all(",
  "  is.na(pm25_array_raw) ==",
  "    array(is.na(pm25_array_raw[, , 1]), dim(pm25_array_raw))",
  ")",
  "",
  "if (na_pct > 0 && na_pct < 20) {",
  "  message(\"Filling NAs with spatial median per time step...\")",
  "  for (t in seq_len(dim(pm25_array_raw)[3])) {",
  "    slice <- pm25_array_raw[, , t]",
  "    if (any(is.na(slice))) {",
  "      slice[is.na(slice)] <- median(slice, na.rm = TRUE)",
  "      pm25_array_raw[, , t] <- slice",
  "    }",
  "  }",
  "} else if (na_pct >= 20) {",
  "  warning(\"High NA percentage — consider cropping the spatial extent to land only.\")",
  "}",
  "",
  "out_fig <- here::here(\"output\", \"figures\")",
  "if (!dir.exists(out_fig)) dir.create(out_fig, recursive = TRUE)",
  "",
  "day1_df <- expand.grid(",
  "  lon = stars::st_dimensions(pm25_raw)$x$offset +",
  "    (seq_len(dim(pm25_raw)[1]) - 0.5) * stars::st_dimensions(pm25_raw)$x$delta,",
  "  lat = stars::st_dimensions(pm25_raw)$y$offset +",
  "    (seq_len(dim(pm25_raw)[2]) - 0.5) * stars::st_dimensions(pm25_raw)$y$delta",
  ")",
  "day1_df$pm25 <- as.vector(pm25_array_raw[, , 1])",
  "",
  "borders_sf <- rnaturalearth::ne_countries(",
  "  scale = \"medium\",",
  "  country = c(",
  "    \"Poland\", \"Germany\", \"Czech Republic\",",
  "    \"Slovakia\", \"Ukraine\", \"Belarus\",",
  "    \"Lithuania\", \"Russia\", \"Austria\"",
  "  ),",
  "  returnclass = \"sf\"",
  ")",
  "poland_sf <- borders_sf %>% dplyr::filter(admin == \"Poland\")",
  "",
  "p_map <- ggplot2::ggplot(day1_df, ggplot2::aes(x = lon, y = lat, fill = pm25)) +",
  "  ggplot2::geom_raster() +",
  "  ggplot2::geom_sf(",
  "    data = borders_sf, inherit.aes = FALSE,",
  "    fill = NA, colour = \"grey25\", linewidth = 0.3",
  "  ) +",
  "  ggplot2::geom_sf(",
  "    data = poland_sf, inherit.aes = FALSE,",
  "    fill = NA, colour = \"black\", linewidth = 0.6",
  "  ) +",
  "  ggplot2::scale_fill_viridis_c(",
  "    name = expression(PM[2.5] ~ \"[\" * mu * g / m^3 * \"]\"),",
  "    na.value = \"grey80\"",
  "  ) +",
  "  ggplot2::coord_sf(",
  "    crs = 4326,",
  "    xlim = range(day1_df$lon), ylim = range(day1_df$lat),",
  "    expand = FALSE",
  "  ) +",
  "  ggplot2::labs(title = paste(\"PM2.5 —\", daily_dates[1])) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"eda_map_day1.png\"), p_map, width = 8, height = 6)",
  "",
  "cities <- data.frame(",
  "  name = c(\"Warszawa\", \"Kraków\", \"Gdańsk\", \"Wrocław\"),",
  "  lon  = c(21.01, 19.94, 18.65, 17.04),",
  "  lat  = c(52.23, 50.06, 54.35, 51.11)",
  ")",
  "",
  "x_coords <- stars::st_dimensions(pm25_raw)$x$offset +",
  "  (seq_len(dim(pm25_raw)[1]) - 0.5) * stars::st_dimensions(pm25_raw)$x$delta",
  "y_coords <- stars::st_dimensions(pm25_raw)$y$offset +",
  "  (seq_len(dim(pm25_raw)[2]) - 0.5) * stars::st_dimensions(pm25_raw)$y$delta",
  "",
  "ts_cities <- do.call(rbind, lapply(seq_len(nrow(cities)), function(r) {",
  "  ix <- which.min(abs(x_coords - cities$lon[r]))",
  "  iy <- which.min(abs(y_coords - cities$lat[r]))",
  "  tibble::tibble(",
  "    name = cities$name[r],",
  "    time = daily_dates,",
  "    pm25 = pm25_array_raw[ix, iy, ]",
  "  )",
  "}))",
  "",
  "cities_sf <- sf::st_as_sf(cities, coords = c(\"lon\", \"lat\"), crs = 4326)",
  "",
  "p_ts <- ggplot2::ggplot(ts_cities, ggplot2::aes(x = time, y = pm25, color = name)) +",
  "  ggplot2::geom_line(alpha = 0.6) +",
  "  ggplot2::labs(",
  "    title = \"PM2.5 daily concentration\",",
  "    y = expression(PM[2.5] ~ \"[\" * mu * g / m^3 * \"]\"),",
  "    x = NULL, color = \"City\"",
  "  ) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"eda_timeseries_cities.png\"), p_ts, width = 10, height = 5, bg = \"white\")",
  "",
  "p_hist <- ggplot2::ggplot(ts_cities, ggplot2::aes(x = pm25, fill = name)) +",
  "  ggplot2::geom_histogram(bins = 50, alpha = 0.5, position = \"identity\") +",
  "  ggplot2::labs(",
  "    title = \"PM2.5 distribution by city\",",
  "    x = expression(PM[2.5] ~ \"[\" * mu * g / m^3 * \"]\")",
  "  ) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"eda_histogram.png\"), p_hist, width = 8, height = 5)",
  "",
  "message(\"Exploratory plots saved to output/figures/\")",
  "",
  "message(\"\\n=== Spatial-statistical hypothesis testing ===\\n\")",
  "",
  "grid_to_sf <- function(arr_2d) {",
  "  grid_coords <- expand.grid(lon = x_coords, lat = y_coords)",
  "  df <- data.frame(",
  "    lon   = grid_coords$lon,",
  "    lat   = grid_coords$lat,",
  "    value = as.vector(arr_2d)",
  "  )",
  "  df <- df[!is.na(df$value), ]",
  "  sf::st_as_sf(df, coords = c(\"lon\", \"lat\"), crs = 4326)",
  "}",
  "",
  "message(\"--- H1: Testing spatial autocorrelation (Global Moran's I) ---\")",
  "",
  "slice_sf <- grid_to_sf(pm25_array_raw[, , 1])",
  "stopifnot(",
  "  \"NA mask varies across days; rebuild slice_sf per day or assert zero NAs\" =",
  "    na_mask_constant_raw",
  ")",
  "knn <- spdep::knearneigh(sf::st_coordinates(slice_sf), k = 8)",
  "nb <- spdep::knn2nb(knn)",
  "lw <- spdep::nb2listw(nb, style = \"W\")",
  "",
  "sample_days <- seq(1, dim(pm25_array_raw)[3], by = 30)",
  "moran_results <- tibble::tibble(",
  "  day_idx = integer(),",
  "  date    = as.Date(character()),",
  "  moran_I = double(),",
  "  p_value = double(),",
  "  z_score = double()",
  ")",
  "",
  "for (d in sample_days) {",
  "  slice_vals <- as.vector(pm25_array_raw[, , d])",
  "  valid <- !is.na(slice_vals)",
  "",
  "  if (sum(valid) < length(slice_vals)) {",
  "    coords_v <- sf::st_coordinates(slice_sf)[valid, ]",
  "    knn_v <- spdep::knearneigh(coords_v, k = min(8, sum(valid) - 1))",
  "    nb_v <- spdep::knn2nb(knn_v)",
  "    lw_v <- spdep::nb2listw(nb_v, style = \"W\")",
  "    mt <- spdep::moran.test(slice_vals[valid], lw_v)",
  "  } else {",
  "    mt <- spdep::moran.test(slice_vals, lw)",
  "  }",
  "",
  "  moran_results <- dplyr::bind_rows(moran_results, tibble::tibble(",
  "    day_idx = d,",
  "    date    = daily_dates[d],",
  "    moran_I = as.numeric(mt$estimate[\"Moran I statistic\"]),",
  "    p_value = mt$p.value,",
  "    z_score = as.numeric(mt$statistic)",
  "  ))",
  "}",
  "",
  "cat(sprintf(\"  Moran's I computed for %d time steps\\n\", nrow(moran_results)))",
  "cat(sprintf(",
  "  \"  Range: [%.3f, %.3f]\\n\",",
  "  min(moran_results$moran_I), max(moran_results$moran_I)",
  "))",
  "cat(sprintf(",
  "  \"  All significant (p < 0.05): %s\\n\",",
  "  ifelse(all(moran_results$p_value < 0.05), \"YES\", \"NO\")",
  "))",
  "cat(sprintf(\"  Mean Moran's I: %.3f\\n\", mean(moran_results$moran_I)))",
  "cat(\"  -> CONCLUSION: Strong positive spatial autocorrelation detected.\\n\")",
  "cat(\"     CNN component is justified — nearby cells carry correlated information.\\n\\n\")",
  "",
  "p_moran_ts <- ggplot2::ggplot(moran_results, ggplot2::aes(x = date, y = moran_I)) +",
  "  ggplot2::geom_line(color = \"steelblue\") +",
  "  ggplot2::geom_point(size = 1, color = \"steelblue\") +",
  "  ggplot2::geom_hline(yintercept = 0, linetype = \"dashed\", color = \"grey50\") +",
  "  ggplot2::labs(",
  "    title = \"Global Moran's I for PM2.5 Over Time\",",
  "    subtitle = \"Consistently positive = persistent spatial autocorrelation\",",
  "    y = \"Moran's I\", x = NULL",
  "  ) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"h1_moran_i_timeseries.png\"), p_moran_ts,",
  "  width = 10, height = 5",
  ")",
  "",
  "repr_day <- sample_days[length(sample_days) %/% 2]",
  "repr_vals <- as.vector(pm25_array_raw[, , repr_day])",
  "valid_repr <- !is.na(repr_vals)",
  "",
  "if (sum(valid_repr) == length(repr_vals)) {",
  "  lag_vals <- spdep::lag.listw(lw, repr_vals)",
  "} else {",
  "  coords_v <- sf::st_coordinates(slice_sf)[valid_repr, ]",
  "  knn_v <- spdep::knearneigh(coords_v, k = min(8, sum(valid_repr) - 1))",
  "  nb_v <- spdep::knn2nb(knn_v)",
  "  lw_v <- spdep::nb2listw(nb_v, style = \"W\")",
  "  lag_vals <- rep(NA, length(repr_vals))",
  "  lag_vals[valid_repr] <- spdep::lag.listw(lw_v, repr_vals[valid_repr])",
  "}",
  "",
  "moran_scatter_df <- tibble::tibble(",
  "  pm25 = repr_vals,",
  "  spatial_lag = lag_vals",
  ") %>% dplyr::filter(!is.na(pm25), !is.na(spatial_lag))",
  "",
  "p_moran_scatter <- ggplot2::ggplot(moran_scatter_df, ggplot2::aes(x = pm25, y = spatial_lag)) +",
  "  ggplot2::geom_point(alpha = 0.3, size = 0.8) +",
  "  ggplot2::geom_smooth(method = \"lm\", color = \"tomato\", se = FALSE) +",
  "  ggplot2::labs(",
  "    title = sprintf(\"Moran Scatterplot — %s\", daily_dates[repr_day]),",
  "    x = expression(PM[2.5] ~ \"[\" * mu * g / m^3 * \"]\"),",
  "    y = expression(\"Spatial lag of \" * PM[2.5])",
  "  ) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"h1_moran_scatterplot.png\"), p_moran_scatter,",
  "  width = 7, height = 6",
  ")",
  "",
  "message(\"--- LISA cluster analysis for representative day ---\")",
  "",
  "if (sum(valid_repr) == length(repr_vals)) {",
  "  lisa <- spdep::localmoran(repr_vals, lw)",
  "  lisa_lag <- spdep::lag.listw(lw, repr_vals)",
  "  lisa_vals <- repr_vals",
  "} else {",
  "  lisa <- spdep::localmoran(repr_vals[valid_repr], lw_v)",
  "  lisa_lag <- spdep::lag.listw(lw_v, repr_vals[valid_repr])",
  "  lisa_vals <- repr_vals[valid_repr]",
  "}",
  "",
  "lisa_df <- tibble::tibble(",
  "  lon   = sf::st_coordinates(slice_sf)[if (sum(valid_repr) < length(repr_vals)) valid_repr else TRUE, 1],",
  "  lat   = sf::st_coordinates(slice_sf)[if (sum(valid_repr) < length(repr_vals)) valid_repr else TRUE, 2],",
  "  Ii    = lisa[, \"Ii\"],",
  "  pval  = lisa[, \"Pr(z != E(Ii))\"],",
  "  value = lisa_vals,",
  "  lag   = lisa_lag",
  ")",
  "",
  "mean_val <- mean(lisa_df$value)",
  "mean_lag <- mean(lisa_df$lag)",
  "",
  "lisa_df <- lisa_df %>%",
  "  dplyr::mutate(",
  "    quadrant = dplyr::case_when(",
  "      value > mean_val & lag > mean_lag ~ \"High-High (hot spot)\",",
  "      value < mean_val & lag < mean_lag ~ \"Low-Low (cold spot)\",",
  "      value > mean_val & lag < mean_lag ~ \"High-Low (outlier)\",",
  "      value < mean_val & lag > mean_lag ~ \"Low-High (outlier)\"",
  "    ),",
  "    quadrant = dplyr::if_else(pval > 0.05, \"Not significant\", quadrant)",
  "  )",
  "",
  "lisa_sf <- sf::st_as_sf(lisa_df, coords = c(\"lon\", \"lat\"), crs = 4326)",
  "",
  "p_lisa <- ggplot2::ggplot() +",
  "  ggplot2::geom_sf(data = lisa_sf, ggplot2::aes(color = quadrant), size = 0.6) +",
  "  ggplot2::scale_color_manual(values = c(",
  "    \"High-High (hot spot)\" = \"#d73027\",",
  "    \"Low-Low (cold spot)\"  = \"#4575b4\",",
  "    \"High-Low (outlier)\"   = \"#fdae61\",",
  "    \"Low-High (outlier)\"   = \"#abd9e9\",",
  "    \"Not significant\"      = \"grey80\"",
  "  )) +",
  "  ggplot2::labs(",
  "    title = sprintf(\"LISA Cluster Map — PM2.5 on %s\", daily_dates[repr_day]),",
  "    subtitle = \"Local indicators of spatial association\",",
  "    color = \"Cluster type\"",
  "  ) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"h1_lisa_clusters.png\"), p_lisa, width = 9, height = 6)",
  "",
  "n_sig <- sum(lisa_df$pval < 0.05)",
  "cat(sprintf(",
  "  \"  Significant LISA clusters: %d / %d cells (%.1f%%) [analytic p, unadjusted]\\n\",",
  "  n_sig, nrow(lisa_df), n_sig / nrow(lisa_df) * 100",
  "))",
  "",
  "message(\"--- H2: Testing temporal autocorrelation (ACF / Ljung-Box) ---\")",
  "",
  "set.seed(42)",
  "random_cells <- sample(which(!is.na(pm25_array_raw[, , 1])), 4)",
  "",
  "random_ts <- lapply(random_cells, function(idx) {",
  "  ij <- arrayInd(idx, .dim = dim(pm25_array_raw)[1:2])",
  "  pm25_array_raw[ij[1], ij[2], ]",
  "})",
  "",
  "city_names <- unique(ts_cities$name)",
  "lb_results <- tibble::tibble(",
  "  location = character(), lb_statistic = double(),",
  "  lb_pvalue = double(), acf_lag1 = double()",
  ")",
  "",
  "for (cn in city_names) {",
  "  city_ts <- ts_cities %>%",
  "    dplyr::filter(name == cn) %>%",
  "    dplyr::pull(pm25)",
  "  city_ts <- city_ts[!is.na(city_ts)]",
  "  if (length(city_ts) < 50) next",
  "",
  "  lb <- Box.test(city_ts, lag = 30, type = \"Ljung-Box\")",
  "  ac <- acf(city_ts, lag.max = 1, plot = FALSE)",
  "",
  "  lb_results <- dplyr::bind_rows(lb_results, tibble::tibble(",
  "    location = cn, lb_statistic = lb$statistic,",
  "    lb_pvalue = lb$p.value, acf_lag1 = ac$acf[2]",
  "  ))",
  "}",
  "",
  "for (i in seq_along(random_ts)) {",
  "  rts <- random_ts[[i]]",
  "  rts <- rts[!is.na(rts)]",
  "  if (length(rts) < 50) next",
  "",
  "  lb <- Box.test(rts, lag = 30, type = \"Ljung-Box\")",
  "  ac <- acf(rts, lag.max = 1, plot = FALSE)",
  "",
  "  lb_results <- dplyr::bind_rows(lb_results, tibble::tibble(",
  "    location = sprintf(\"Random cell %d\", i), lb_statistic = lb$statistic,",
  "    lb_pvalue = lb$p.value, acf_lag1 = ac$acf[2]",
  "  ))",
  "}",
  "",
  "cat(\"  Ljung-Box test results (H0: no temporal autocorrelation):\\n\")",
  "print(as.data.frame(lb_results), row.names = FALSE)",
  "cat(sprintf(",
  "  \"\\n  All significant (p < 0.05): %s\\n\",",
  "  ifelse(all(lb_results$lb_pvalue < 0.05), \"YES\", \"NO\")",
  "))",
  "cat(sprintf(\"  Mean lag-1 ACF: %.3f\\n\", mean(lb_results$acf_lag1)))",
  "cat(\"  -> CONCLUSION: Strong temporal autocorrelation at all locations.\\n\")",
  "cat(\"     LSTM component is justified — past values predict future values.\\n\\n\")",
  "",
  "city_example <- ts_cities %>%",
  "  dplyr::filter(name == city_names[1]) %>%",
  "  dplyr::pull(pm25)",
  "city_example <- city_example[!is.na(city_example)]",
  "acf_obj <- acf(city_example, lag.max = 60, plot = FALSE)",
  "",
  "p_acf <- ggplot2::ggplot(",
  "  tibble::tibble(lag = acf_obj$lag[-1], acf = acf_obj$acf[-1]),",
  "  ggplot2::aes(x = lag, y = acf)",
  ") +",
  "  ggplot2::geom_hline(yintercept = 0, color = \"grey50\") +",
  "  ggplot2::geom_hline(",
  "    yintercept = c(-1, 1) * qnorm(0.975) / sqrt(length(city_example)),",
  "    linetype = \"dashed\", color = \"blue\"",
  "  ) +",
  "  ggplot2::geom_segment(ggplot2::aes(xend = lag, yend = 0), color = \"steelblue\") +",
  "  ggplot2::labs(",
  "    title = sprintf(\"Autocorrelation Function — PM2.5 in %s\", city_names[1]),",
  "    subtitle = \"Dashed lines = 95% confidence interval for white noise\",",
  "    x = \"Lag [days]\", y = \"ACF\"",
  "  ) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"h2_acf_city.png\"), p_acf, width = 8, height = 5)",
  "",
  "message(\"--- H3: Testing non-stationarity of spatial patterns ---\")",
  "",
  "lags_to_test <- c(1, 7, 14, 30, 60, 90)",
  "field_corrs <- tibble::tibble(lag = integer(), mean_cor = double(), sd_cor = double())",
  "",
  "n_samples_corr <- min(100, dim(pm25_array_raw)[3] %/% 2)",
  "sample_starts <- sample(",
  "  1:(dim(pm25_array_raw)[3] - max(lags_to_test)),",
  "  n_samples_corr",
  ")",
  "",
  "for (lg in lags_to_test) {",
  "  cors <- sapply(sample_starts, function(t0) {",
  "    v1 <- as.vector(pm25_array_raw[, , t0])",
  "    v2 <- as.vector(pm25_array_raw[, , t0 + lg])",
  "    valid <- !is.na(v1) & !is.na(v2)",
  "    cor(v1[valid], v2[valid])",
  "  })",
  "",
  "  field_corrs <- dplyr::bind_rows(field_corrs, tibble::tibble(",
  "    lag = lg, mean_cor = mean(cors), sd_cor = sd(cors)",
  "  ))",
  "}",
  "",
  "p_field_corr <- ggplot2::ggplot(field_corrs, ggplot2::aes(x = lag, y = mean_cor)) +",
  "  ggplot2::geom_line(color = \"steelblue\", linewidth = 1) +",
  "  ggplot2::geom_point(size = 3, color = \"steelblue\") +",
  "  ggplot2::geom_ribbon(ggplot2::aes(ymin = mean_cor - sd_cor, ymax = pmin(mean_cor + sd_cor, 1)),",
  "    alpha = 0.2, fill = \"steelblue\"",
  "  ) +",
  "  ggplot2::labs(",
  "    title = \"Spatial Field Correlation at Increasing Time Lags\",",
  "    subtitle = \"Decay in correlation = spatial patterns change over time\",",
  "    x = \"Time lag [days]\", y = \"Pearson r between PM2.5 grids\"",
  "  ) +",
  "  ggplot2::scale_x_continuous(breaks = lags_to_test) +",
  "  ggplot2::ylim(0, 1) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"h3_field_correlation_decay.png\"), p_field_corr,",
  "  width = 8, height = 5",
  ")",
  "",
  "cat(sprintf(\"\\n  Spatial field correlation decay:\\n\"))",
  "for (i in seq_len(nrow(field_corrs))) {",
  "  cat(sprintf(",
  "    \"    Lag %3d days: r = %.3f (±%.3f)\\n\",",
  "    field_corrs$lag[i], field_corrs$mean_cor[i], field_corrs$sd_cor[i]",
  "  ))",
  "}",
  "",
  "cat(\"\\n  -> CONCLUSION: Spatial patterns are non-stationary.\\n\")",
  "cat(\"     Correlation decays with time lag — a static spatial model is insufficient.\\n\")",
  "cat(\"     A spatio-temporal architecture (CNN-LSTM) is methodologically justified.\\n\\n\")",
  "",
  "cat(\"=============================================================\\n\")",
  "cat(\"  HYPOTHESIS TESTING SUMMARY\\n\")",
  "cat(\"=============================================================\\n\")",
  "cat(\"  H1: Spatial autocorrelation exists?       YES (Moran's I)\\n\")",
  "cat(\"      -> CNN justified for spatial feature extraction\\n\\n\")",
  "cat(\"  H2: Temporal autocorrelation exists?       YES (Ljung-Box)\\n\")",
  "cat(\"      -> LSTM justified for temporal sequence modelling\\n\\n\")",
  "cat(\"  H3: Spatial patterns change over time?     YES (field corr. decay)\\n\")",
  "cat(\"      -> Spatio-temporal model justified over static spatial model\\n\")",
  "cat(\"=============================================================\\n\\n\")",
  "",
  "message(\"Hypothesis testing plots saved to output/figures/\")",
  "",
  "pm25_array_log <- log1p(pm25_array_raw)",
  "",
  "WINDOW_SIZE <- 30",
  "",
  "n_days_all <- dim(pm25_array_log)[3]",
  "train_end_samples <- floor((n_days_all - WINDOW_SIZE) * 0.70)",
  "train_day_end <- train_end_samples + WINDOW_SIZE",
  "",
  "pm25_min <- min(pm25_array_log[, , 1:train_day_end], na.rm = TRUE)",
  "pm25_max <- max(pm25_array_log[, , 1:train_day_end], na.rm = TRUE)",
  "pm25_scaled <- (pm25_array_log - pm25_min) / (pm25_max - pm25_min)",
  "",
  "message(sprintf(",
  "  \"Log-scaling (train days 1..%d of %d): log1p min=%.4f, max=%.4f (raw train: %.2f .. %.2f)\",",
  "  train_day_end, n_days_all,",
  "  pm25_min, pm25_max,",
  "  min(pm25_array_raw[, , 1:train_day_end], na.rm = TRUE),",
  "  max(pm25_array_raw[, , 1:train_day_end], na.rm = TRUE)",
  "))",
  "",
  "times <- daily_dates",
  "doy <- as.numeric(format(times, \"%j\"))",
  "dow <- as.POSIXlt(times)$wday",
  "mon <- as.numeric(format(times, \"%m\"))",
  "n_days <- length(times)",
  "",
  "sin_doy <- sin(2 * pi * doy / 365)",
  "cos_doy <- cos(2 * pi * doy / 365)",
  "sin_doy_2h <- sin(4 * pi * doy / 365)",
  "cos_doy_2h <- cos(4 * pi * doy / 365)",
  "",
  "sin_dow <- sin(2 * pi * dow / 7)",
  "cos_dow <- cos(2 * pi * dow / 7)",
  "",
  "weekend <- as.integer(dow %in% c(0, 6))",
  "",
  "heating_season <- as.integer(mon %in% c(10, 11, 12, 1, 2, 3, 4))",
  "",
  "polish_holidays <- as.Date(c(",
  "  \"2018-01-01\", \"2018-01-06\", \"2018-05-01\", \"2018-05-03\", \"2018-08-15\",",
  "  \"2018-11-01\", \"2018-11-11\", \"2018-12-25\", \"2018-12-26\",",
  "  \"2018-04-01\", \"2018-04-02\", \"2018-05-20\", \"2018-05-31\",",
  "  \"2019-01-01\", \"2019-01-06\", \"2019-05-01\", \"2019-05-03\", \"2019-08-15\",",
  "  \"2019-11-01\", \"2019-11-11\", \"2019-12-25\", \"2019-12-26\",",
  "  \"2019-04-21\", \"2019-04-22\", \"2019-06-09\", \"2019-06-20\",",
  "  \"2020-01-01\", \"2020-01-06\", \"2020-05-01\", \"2020-05-03\", \"2020-08-15\",",
  "  \"2020-11-01\", \"2020-11-11\", \"2020-12-25\", \"2020-12-26\",",
  "  \"2020-04-12\", \"2020-04-13\", \"2020-05-31\", \"2020-06-11\",",
  "  \"2021-01-01\", \"2021-01-06\", \"2021-05-01\", \"2021-05-03\", \"2021-08-15\",",
  "  \"2021-11-01\", \"2021-11-11\", \"2021-12-25\", \"2021-12-26\",",
  "  \"2021-04-04\", \"2021-04-05\", \"2021-05-23\", \"2021-06-03\",",
  "  \"2022-01-01\", \"2022-01-06\", \"2022-05-01\", \"2022-05-03\", \"2022-08-15\",",
  "  \"2022-11-01\", \"2022-11-11\", \"2022-12-25\", \"2022-12-26\",",
  "  \"2022-04-17\", \"2022-04-18\", \"2022-06-05\", \"2022-06-16\"",
  "))",
  "years_in_data <- sort(unique(as.numeric(format(times, \"%Y\"))))",
  "if (any(!years_in_data %in% 2018:2022)) {",
  "  warning(sprintf(",
  "    \"Data contains years outside the hardcoded Polish holiday range 2018–2022: %s. Extend polish_holidays in 02_data_prep.R.\",",
  "    paste(setdiff(years_in_data, 2018:2022), collapse = \", \")",
  "  ))",
  "}",
  "holiday <- as.integer(times %in% polish_holidays)",
  "",
  "non_working <- (weekend == 1) | (holiday == 1)",
  "bridge <- logical(n_days)",
  "if (n_days >= 3) {",
  "  for (k in 2:(n_days - 1)) {",
  "    if (!non_working[k] && non_working[k - 1] && non_working[k + 1]) {",
  "      bridge[k] <- TRUE",
  "    }",
  "  }",
  "}",
  "extended_off <- non_working | bridge",
  "long_weekend_vec <- logical(n_days)",
  "runs <- rle(extended_off)",
  "pos <- 1L",
  "for (k in seq_along(runs$lengths)) {",
  "  len <- runs$lengths[k]",
  "  if (isTRUE(runs$values[k]) && len >= 3) {",
  "    long_weekend_vec[pos:(pos + len - 1L)] <- TRUE",
  "  }",
  "  pos <- pos + len",
  "}",
  "long_weekend <- as.integer(long_weekend_vec)",
  "",
  "linear_trend <- (seq_len(n_days) - 1) / max(train_day_end - 1, 1)",
  "",
  "nx <- dim(pm25_scaled)[1]",
  "ny <- dim(pm25_scaled)[2]",
  "nt <- dim(pm25_scaled)[3]",
  "stopifnot(nt == n_days)",
  "",
  "channel_series <- list(",
  "  sin_doy        = sin_doy,",
  "  cos_doy        = cos_doy,",
  "  sin_doy_2h     = sin_doy_2h,",
  "  cos_doy_2h     = cos_doy_2h,",
  "  sin_dow        = sin_dow,",
  "  cos_dow        = cos_dow,",
  "  weekend        = as.numeric(weekend),",
  "  holiday        = as.numeric(holiday),",
  "  heating_season = as.numeric(heating_season),",
  "  long_weekend   = as.numeric(long_weekend),",
  "  linear_trend   = linear_trend",
  ")",
  "",
  "spatial_path <- here::here(\"data\", \"processed\", \"spatial_features.rds\")",
  "stopifnot(",
  "  \"spatial_features.rds missing — run `Rscript code/01_data_acquisition.R` first\" =",
  "    file.exists(spatial_path)",
  ")",
  "spatial_features <- readRDS(spatial_path)",
  "stopifnot(",
  "  \"spatial_features elevation shape mismatches CAMS grid\" =",
  "    all(dim(spatial_features$elevation) == c(nx, ny)),",
  "  \"spatial_features log_pop_density shape mismatches CAMS grid\" =",
  "    all(dim(spatial_features$log_pop_density) == c(nx, ny))",
  ")",
  "",
  "static_spatial <- list(",
  "  elevation       = spatial_features$elevation,",
  "  log_pop_density = spatial_features$log_pop_density",
  ")",
  "",
  "channel_names <- c(\"pm25_scaled\", names(channel_series), names(static_spatial))",
  "n_channels <- length(channel_names)",
  "",
  "pm25_multi <- array(dim = c(nx, ny, nt, n_channels))",
  "pm25_multi[, , , 1] <- pm25_scaled",
  "for (c_idx in seq_along(channel_series)) {",
  "  pm25_multi[, , , c_idx + 1L] <- rep(channel_series[[c_idx]], each = nx * ny)",
  "}",
  "static_offset <- 1L + length(channel_series)",
  "for (c_idx in seq_along(static_spatial)) {",
  "  pm25_multi[, , , static_offset + c_idx] <-",
  "    rep(as.vector(static_spatial[[c_idx]]), times = nt)",
  "}",
  "",
  "message(sprintf(",
  "  \"Multi-channel array: %s (x, y, time, channels)\",",
  "  paste(dim(pm25_multi), collapse = \" x \")",
  "))",
  "message(sprintf(",
  "  \"Channels (%d): %s\", n_channels,",
  "  paste(channel_names, collapse = \", \")",
  "))",
  "message(sprintf(",
  "  \"  holiday days: %d | weekend days: %d | heating-season days: %d | long-weekend days: %d\",",
  "  sum(holiday), sum(weekend), sum(heating_season), sum(long_weekend)",
  "))",
  "message(sprintf(",
  "  \"  elevation scale: raw [%.0f, %.0f] m | pop density max: %.0f /km²\",",
  "  spatial_features$meta$elev_min_m,",
  "  spatial_features$meta$elev_max_m,",
  "  spatial_features$meta$pop_density_max",
  "))",
  "",
  "n_time <- dim(pm25_multi)[3]",
  "n_samples <- n_time - WINDOW_SIZE",
  "",
  "message(sprintf(",
  "  \"Creating sliding window indices (T=%d, n=%d samples)...\",",
  "  WINDOW_SIZE, n_samples",
  "))",
  "",
  "Y <- array(dim = c(n_samples, nx, ny))",
  "for (i in seq_len(n_samples)) {",
  "  Y[i, , ] <- pm25_multi[, , i + WINDOW_SIZE, 1]",
  "}",
  "",
  "message(sprintf(\"  Y: %s\", paste(dim(Y), collapse = \" x \")))",
  "message(sprintf(",
  "  \"  X will be sliced on-the-fly from source array (%.1f GB saved)\",",
  "  n_samples * WINDOW_SIZE * nx * ny * n_channels * 8 / 1e9",
  "))",
  "",
  "train_end <- floor(n_samples * 0.70)",
  "val_start <- train_end + 1L",
  "val_end <- floor(n_samples * 0.85)",
  "test_start <- val_end + 1L",
  "",
  "stopifnot(",
  "  \"Empty val split; n_samples too small\"  = val_start <= val_end,",
  "  \"Empty test split; n_samples too small\" = test_start <= n_samples",
  ")",
  "",
  "split_idx <- list(",
  "  train = 1:train_end,",
  "  val   = val_start:val_end,",
  "  test  = test_start:n_samples",
  ")",
  "",
  "message(sprintf(",
  "  \"Split: train=%d, val=%d, test=%d\",",
  "  length(split_idx$train),",
  "  length(split_idx$val),",
  "  length(split_idx$test)",
  "))",
  "",
  "tensors <- list(",
  "  source_array = pm25_multi,",
  "  Y_train = Y[split_idx$train, , ],",
  "  Y_val = Y[split_idx$val, , ],",
  "  Y_test = Y[split_idx$test, , ],",
  "  meta = list(",
  "    window_size      = WINDOW_SIZE,",
  "    n_channels       = n_channels,",
  "    channel_names    = channel_names,",
  "    log_transform    = TRUE,",
  "    pm25_min         = pm25_min,",
  "    pm25_max         = pm25_max,",
  "    grid_dim         = c(nx = nx, ny = ny),",
  "    times            = times,",
  "    split_idx        = split_idx,",
  "    cities           = cities_sf,",
  "    spatial_channels = spatial_features$meta",
  "  )",
  ")",
  "",
  "proc_dir <- here::here(\"data\", \"processed\")",
  "if (!dir.exists(proc_dir)) dir.create(proc_dir, recursive = TRUE)",
  "",
  "saveRDS(tensors, file.path(proc_dir, \"pm25_tensors.rds\"))",
  "message(sprintf(\"Saved to %s/pm25_tensors.rds\", proc_dir))",
  "",
  "cat(\"\\n=== Data preparation complete ===\\n\")",
  "cat(sprintf(\"  Grid:     %d x %d (lon x lat)\\n\", nx, ny))",
  "cat(sprintf(\"  Days:     %d\\n\", nt))",
  "cat(sprintf(\"  Window:   %d days\\n\", WINDOW_SIZE))",
  "cat(sprintf(",
  "  \"  Channels: %d (%s)\\n\", n_channels,",
  "  paste(tensors$meta$channel_names, collapse = \", \")",
  "))",
  "cat(sprintf(",
  "  \"  Samples:  %d train / %d val / %d test\\n\",",
  "  length(split_idx$train),",
  "  length(split_idx$val),",
  "  length(split_idx$test)",
  "))",
  "cat(sprintf(",
  "  \"  X shape:  (n, %d, %d, %d, %d)  [sliced on-the-fly]\\n\",",
  "  WINDOW_SIZE, nx, ny, n_channels",
  "))",
  "cat(sprintf(\"  Y shape:  (n, %d, %d)\\n\", nx, ny))",
  "cat(sprintf(",
  "  \"  Source array: %.1f MB\\n\",",
  "  object.size(tensors$source_array) / 1e6",
  "))"
))
files_written <- c(files_written, "code/02_data_prep.R")

write_code_file("code/03_baselines.R", c(
  "library(tidyverse)",
  "library(here)",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "src     <- tensors$source_array",
  "",
  "pm_scaled <- src[, , , 1]",
  "nx <- dim(pm_scaled)[1]",
  "ny <- dim(pm_scaled)[2]",
  "nt <- dim(pm_scaled)[3]",
  "",
  "test_idx <- meta$split_idx$test",
  "win      <- meta$window_size",
  "n_test   <- length(test_idx)",
  "",
  "inv_scale <- function(x) {",
  "  unscaled <- x * (meta$pm25_max - meta$pm25_min) + meta$pm25_min",
  "  if (isTRUE(meta$log_transform)) expm1(unscaled) else unscaled",
  "}",
  "",
  "target_days <- test_idx + win",
  "fit_end     <- target_days[1] - 1",
  "",
  "cat(sprintf(\"Fit days 1..%d; %d test targets %d..%d\\n\",",
  "            fit_end, n_test, target_days[1], target_days[n_test]))",
  "",
  "actuals_scaled <- array(NA_real_, dim = c(n_test, nx, ny))",
  "for (k in seq_len(n_test)) actuals_scaled[k, , ] <- pm_scaled[, , target_days[k]]",
  "actuals <- inv_scale(actuals_scaled)",
  "",
  "y_fit   <- pm_scaled[, , 1:fit_end]",
  "Tfit    <- dim(y_fit)[3]",
  "y_t     <- y_fit[, , 2:Tfit]",
  "y_lag   <- y_fit[, , 1:(Tfit - 1)]",
  "",
  "mean_t   <- apply(y_t,   c(1, 2), mean)",
  "mean_lag <- apply(y_lag, c(1, 2), mean)",
  "cov_xy   <- apply(y_t * y_lag, c(1, 2), mean) - mean_t * mean_lag",
  "var_x    <- apply(y_lag^2,     c(1, 2), mean) - mean_lag^2",
  "",
  "stopifnot(\"Zero-variance cell in AR(1) fit window\" = all(var_x > 0))",
  "phi_ar   <- cov_xy / var_x",
  "alpha_ar <- mean_t - phi_ar * mean_lag",
  "",
  "ar1_preds_scaled <- array(NA_real_, dim = c(n_test, nx, ny))",
  "for (k in seq_len(n_test)) {",
  "  y_prev <- pm_scaled[, , target_days[k] - 1]",
  "  ar1_preds_scaled[k, , ] <- alpha_ar + phi_ar * y_prev",
  "}",
  "ar1_preds <- pmax(inv_scale(ar1_preds_scaled), 0)",
  "",
  "rmse_ar1 <- sqrt(mean((ar1_preds - actuals)^2, na.rm = TRUE))",
  "mae_ar1  <- mean(abs(ar1_preds - actuals), na.rm = TRUE)",
  "",
  "cat(sprintf(\"\\nAR(1) per cell:  RMSE = %.3f µg/m³, MAE = %.3f\\n\", rmse_ar1, mae_ar1))",
  "cat(sprintf(\"  mean phi = %.3f (persistence-equivalent = 1)\\n\",",
  "            mean(phi_ar, na.rm = TRUE)))",
  "",
  "spatial_lag <- function(mat) {",
  "  nx <- nrow(mat); ny <- ncol(mat)",
  "  padded <- matrix(NA_real_, nx + 2, ny + 2)",
  "  padded[2:(nx + 1), 2:(ny + 1)] <- mat",
  "  total <- matrix(0, nx, ny)",
  "  count <- matrix(0L, nx, ny)",
  "  for (di in -1:1) for (dj in -1:1) {",
  "    if (di == 0 && dj == 0) next",
  "    sh <- padded[(2 + di):(nx + 1 + di), (2 + dj):(ny + 1 + dj)]",
  "    ok <- !is.na(sh)",
  "    total <- total + ifelse(ok, sh, 0)",
  "    count <- count + ok",
  "  }",
  "  total / count",
  "}",
  "",
  "cat(\"\\nComputing spatial lags ...\\n\")",
  "Wy <- array(NA_real_, dim = c(nx, ny, nt))",
  "for (t in seq_len(nt)) Wy[, , t] <- spatial_lag(pm_scaled[, , t])",
  "",
  "y_train    <- as.vector(pm_scaled[, , 2:fit_end])",
  "ylag_train <- as.vector(pm_scaled[, , 1:(fit_end - 1)])",
  "wylag_train <- as.vector(Wy[, , 1:(fit_end - 1)])",
  "",
  "fit_star <- lm(y_train ~ ylag_train + wylag_train)",
  "coefs    <- coef(fit_star)",
  "",
  "cat(\"STAR coefficients:\\n\"); print(coefs)",
  "",
  "star_preds_scaled <- array(NA_real_, dim = c(n_test, nx, ny))",
  "for (k in seq_len(n_test)) {",
  "  y_prev  <- pm_scaled[, , target_days[k] - 1]",
  "  wy_prev <- Wy[, , target_days[k] - 1]",
  "  star_preds_scaled[k, , ] <- coefs[1] + coefs[2] * y_prev + coefs[3] * wy_prev",
  "}",
  "star_preds <- pmax(inv_scale(star_preds_scaled), 0)",
  "",
  "rmse_star <- sqrt(mean((star_preds - actuals)^2, na.rm = TRUE))",
  "mae_star  <- mean(abs(star_preds - actuals), na.rm = TRUE)",
  "cat(sprintf(\"\\nSTAR (pooled):   RMSE = %.3f µg/m³, MAE = %.3f\\n\", rmse_star, mae_star))",
  "",
  "saveRDS(",
  "  list(",
  "    ar1  = list(predictions = ar1_preds,  rmse = rmse_ar1,  mae = mae_ar1,",
  "                phi = phi_ar, alpha = alpha_ar),",
  "    star = list(predictions = star_preds, rmse = rmse_star, mae = mae_star,",
  "                coefs = coefs),",
  "    actuals = actuals",
  "  ),",
  "  here::here(\"data\", \"processed\", \"baselines_results.rds\")",
  ")",
  "",
  "cat(\"\\nSaved: data/processed/baselines_results.rds\\n\")"
))
files_written <- c(files_written, "code/03_baselines.R")

write_code_file("code/04_cnn_lstm_skip.R", c(
  "library(here)",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "",
  "cat(\"=== Loaded data ===\\n\")",
  "cat(sprintf(\"  Source array: %s\\n\", paste(dim(tensors$source_array), collapse = \" x \")))",
  "cat(sprintf(\"  Y_train: %s\\n\", paste(dim(tensors$Y_train), collapse = \" x \")))",
  "cat(sprintf(\"  Y_val:   %s\\n\", paste(dim(tensors$Y_val), collapse = \" x \")))",
  "cat(sprintf(\"  Y_test:  %s\\n\", paste(dim(tensors$Y_test), collapse = \" x \")))",
  "cat(sprintf(\"  Grid: %d x %d, Window: %d, Channels: %d\\n\",",
  "            meta$grid_dim[[1]], meta$grid_dim[[2]],",
  "            meta$window_size, meta$n_channels))",
  "",
  "tp  <- arch_hparams(\"spatial\")",
  "dls <- make_dataloaders(tensors, meta, batch_size = tp$batch_size)",
  "",
  "batch <- dls$train$.iter()$.next()",
  "cat(sprintf(\"\\nBatch X shape: %s  (batch, T, C, H, W)\\n\",",
  "            paste(batch$x$shape, collapse = \" x \")))",
  "cat(sprintf(\"Batch Y shape: %s  (batch, 1, H, W)\\n\",",
  "            paste(batch$y$shape, collapse = \" x \")))",
  "",
  "cnn_lstm <- torch::nn_module(",
  "  \"cnn_lstm\",",
  "  initialize = function(in_channels, hidden_dim, lstm_layers, grid_h, grid_w) {",
  "    self$encoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(in_channels, 32, kernel_size = 3, stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(32),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(32, 64, kernel_size = 3, stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(64),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(64, 64, kernel_size = 3, stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(64),",
  "      torch::nn_relu()",
  "    )",
  "    enc_h <- ceiling(grid_h / 8)",
  "    enc_w <- ceiling(grid_w / 8)",
  "    self$enc_fc <- torch::nn_linear(64 * enc_h * enc_w, hidden_dim)",
  "",
  "    self$lstm <- torch::nn_lstm(",
  "      input_size  = hidden_dim,",
  "      hidden_size = hidden_dim,",
  "      num_layers  = lstm_layers,",
  "      batch_first = TRUE,",
  "      dropout     = if (lstm_layers > 1) 0.2 else 0",
  "    )",
  "",
  "    self$dec_fc    <- torch::nn_linear(hidden_dim, 64 * enc_h * enc_w)",
  "    self$dec_enc_h <- enc_h",
  "    self$dec_enc_w <- enc_w",
  "    self$decoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(64, 32, kernel_size = 3, padding = 1),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(32, 1,  kernel_size = 3, padding = 1)",
  "    )",
  "    self$upsample <- torch::nn_upsample(size = c(grid_h, grid_w), mode = \"bilinear\",",
  "                                 align_corners = FALSE)",
  "  },",
  "  forward = function(x) {",
  "    b  <- x$shape[1]",
  "    tt <- x$shape[2]",
  "",
  "    skip <- x[, tt, 1, , ]$unsqueeze(2)",
  "",
  "    x_flat <- x$reshape(c(b * tt, x$shape[3], x$shape[4], x$shape[5]))",
  "    feat   <- self$encoder(x_flat)",
  "    feat   <- feat$view(c(b * tt, -1))",
  "    feat   <- self$enc_fc(feat)",
  "    feat   <- feat$view(c(b, tt, -1))",
  "",
  "    lstm_out <- self$lstm(feat)",
  "    last_h   <- lstm_out[[1]][, tt, ]",
  "",
  "    out   <- self$dec_fc(last_h)",
  "    out   <- out$view(c(b, 64, self$dec_enc_h, self$dec_enc_w))",
  "    delta <- self$decoder(out)",
  "    delta <- self$upsample(delta)",
  "",
  "    skip + delta",
  "  }",
  ")",
  "",
  "HIDDEN_DIM  <- 128L",
  "LSTM_LAYERS <- 1L",
  "grid_h      <- as.integer(dim(tensors$source_array)[1])",
  "grid_w      <- as.integer(dim(tensors$source_array)[2])",
  "",
  "cat(sprintf(\"\\n=== Model config ===\\n\"))",
  "cat(sprintf(\"  hidden_dim:  %d\\n\", HIDDEN_DIM))",
  "cat(sprintf(\"  lstm_layers: %d\\n\", LSTM_LAYERS))",
  "cat(sprintf(\"  lr:          %.4f\\n\", tp$lr))",
  "cat(sprintf(\"  epochs:      %d\\n\", EPOCHS))",
  "cat(sprintf(\"  batch_size:  %d\\n\", tp$batch_size))",
  "cat(sprintf(\"  grid:        %d x %d\\n\", grid_h, grid_w))",
  "",
  "log_path <- here::here(\"output\", \"training_log_cnnlstm.csv\")",
  "fitted_model <- load_or_train(",
  "  cnn_lstm,",
  "  hparams  = list(in_channels = meta$n_channels,",
  "                  hidden_dim  = HIDDEN_DIM,",
  "                  lstm_layers = LSTM_LAYERS,",
  "                  grid_h      = grid_h,",
  "                  grid_w      = grid_w),",
  "  dls            = dls,",
  "  log_path       = log_path,",
  "  model_filename = \"cnn_lstm_final.pt\",",
  "  lr             = tp$lr,",
  "  accelerator    = tp$accelerator,",
  "  fig_path       = here::here(\"output\", \"figures\", \"training_loss_cnnlstm.png\"),",
  "  fig_title      = \"CNN-LSTM Training Loss (MSE on log1p-scaled targets)\"",
  ")",
  "",
  "inv_scale    <- make_inv_scale(meta)",
  "rmse_persist <- sqrt(mean((persistence_pm25(tensors, meta, inv_scale)",
  "                           - inv_scale(tensors$Y_test))^2, na.rm = TRUE))",
  "",
  "result <- evaluate_and_save(",
  "  fitted_model, dls$test, tensors, meta,",
  "  results_path = here::here(\"data\", \"processed\", \"cnn_lstm_skip_test_results.rds\"),",
  "  label        = \"CNN-LSTM (skip)\"",
  ")",
  "",
  "cat(sprintf(\"\\n=== Model vs persistence ===\\n\"))",
  "cat(sprintf(\"  RMSE: %.2f µg/m³  (persistence: %.2f)\\n\",",
  "            result$rmse, rmse_persist))",
  "",
  "if (result$rmse >= rmse_persist) {",
  "  warning(sprintf(",
  "    \"Model does not beat persistence (RMSE %.2f >= %.2f).\",",
  "    result$rmse, rmse_persist))",
  "}"
))
files_written <- c(files_written, "code/04_cnn_lstm_skip.R")

write_code_file("code/05_cnn_lstm_noskip.R", c(
  "library(here)",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "",
  "cat(\"=== CNN-LSTM (no skip) ===\\n\")",
  "cat(sprintf(\"  Grid: %d x %d, Window: %d, Channels: %d\\n\",",
  "            meta$grid_dim[[1]], meta$grid_dim[[2]],",
  "            meta$window_size, meta$n_channels))",
  "",
  "tp  <- arch_hparams(\"spatial\")",
  "dls <- make_dataloaders(tensors, meta, batch_size = tp$batch_size)",
  "",
  "cnn_lstm_noskip <- torch::nn_module(",
  "  \"cnn_lstm_noskip\",",
  "  initialize = function(in_channels, hidden_dim, lstm_layers, grid_h, grid_w) {",
  "    self$encoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(in_channels, 32, 3, stride = 2, padding = 1), torch::nn_batch_norm2d(32), torch::nn_relu(),",
  "      torch::nn_conv2d(32, 64, 3, stride = 2, padding = 1), torch::nn_batch_norm2d(64), torch::nn_relu(),",
  "      torch::nn_conv2d(64, 64, 3, stride = 2, padding = 1), torch::nn_batch_norm2d(64), torch::nn_relu()",
  "    )",
  "    enc_h <- ceiling(grid_h / 8); enc_w <- ceiling(grid_w / 8)",
  "    self$enc_fc <- torch::nn_linear(64 * enc_h * enc_w, hidden_dim)",
  "    self$lstm <- torch::nn_lstm(hidden_dim, hidden_dim, num_layers = lstm_layers,",
  "                         batch_first = TRUE,",
  "                         dropout = if (lstm_layers > 1) 0.2 else 0)",
  "    self$dec_fc <- torch::nn_linear(hidden_dim, 64 * enc_h * enc_w)",
  "    self$dec_enc_h <- enc_h; self$dec_enc_w <- enc_w",
  "    self$decoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(64, 32, 3, padding = 1), torch::nn_relu(),",
  "      torch::nn_conv2d(32, 1, 3, padding = 1)",
  "    )",
  "    self$upsample <- torch::nn_upsample(size = c(grid_h, grid_w), mode = \"bilinear\",",
  "                                 align_corners = FALSE)",
  "  },",
  "  forward = function(x) {",
  "    b <- x$shape[1]; tt <- x$shape[2]",
  "    x_flat <- x$reshape(c(b * tt, x$shape[3], x$shape[4], x$shape[5]))",
  "    feat <- self$encoder(x_flat)",
  "    feat <- feat$view(c(b * tt, -1))",
  "    feat <- self$enc_fc(feat)",
  "    feat <- feat$view(c(b, tt, -1))",
  "    lstm_out <- self$lstm(feat)",
  "    last_h <- lstm_out[[1]][, tt, ]",
  "    out <- self$dec_fc(last_h)",
  "    out <- out$view(c(b, 64, self$dec_enc_h, self$dec_enc_w))",
  "    out <- self$decoder(out)",
  "    self$upsample(out)",
  "  }",
  ")",
  "",
  "HIDDEN_DIM  <- 128L",
  "LSTM_LAYERS <- 1L",
  "grid_h      <- as.integer(dim(tensors$source_array)[1])",
  "grid_w      <- as.integer(dim(tensors$source_array)[2])",
  "",
  "cat(sprintf(\"  hidden_dim: %d, lstm_layers: %d, epochs: %d, batch: %d\\n\",",
  "            HIDDEN_DIM, LSTM_LAYERS, EPOCHS, tp$batch_size))",
  "",
  "log_path <- here::here(\"output\", \"training_log_cnnlstm_noskip.csv\")",
  "fitted_noskip <- load_or_train(",
  "  cnn_lstm_noskip,",
  "  hparams  = list(in_channels = meta$n_channels,",
  "                  hidden_dim  = HIDDEN_DIM,",
  "                  lstm_layers = LSTM_LAYERS,",
  "                  grid_h      = grid_h,",
  "                  grid_w      = grid_w),",
  "  dls            = dls,",
  "  log_path       = log_path,",
  "  model_filename = \"cnn_lstm_noskip_final.pt\",",
  "  lr             = tp$lr,",
  "  accelerator    = tp$accelerator,",
  "  fig_path       = here::here(\"output\", \"figures\", \"training_loss_cnnlstm_noskip.png\"),",
  "  fig_title      = \"CNN-LSTM (no skip) — training loss\"",
  ")",
  "",
  "evaluate_and_save(",
  "  fitted_noskip, dls$test, tensors, meta,",
  "  results_path = here::here(\"data\", \"processed\", \"noskip_test_results.rds\"),",
  "  label        = \"CNN-LSTM (no skip)\"",
  ")"
))
files_written <- c(files_written, "code/05_cnn_lstm_noskip.R")

write_code_file("code/06_lstm_only_skip.R", c(
  "library(here)",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "",
  "cat(\"=== Per-cell LSTM (no spatial features) ===\\n\")",
  "cat(sprintf(\"  Grid: %d x %d, Window: %d, Channels: %d\\n\",",
  "            meta$grid_dim[[1]], meta$grid_dim[[2]],",
  "            meta$window_size, meta$n_channels))",
  "",
  "tp  <- arch_hparams(\"percell\")",
  "dls <- make_dataloaders(tensors, meta, batch_size = tp$batch_size)",
  "",
  "lstm_only <- torch::nn_module(",
  "  \"lstm_only\",",
  "  initialize = function(in_channels, hidden_dim, lstm_layers) {",
  "    self$lstm <- torch::nn_lstm(",
  "      input_size  = in_channels,",
  "      hidden_size = hidden_dim,",
  "      num_layers  = lstm_layers,",
  "      batch_first = TRUE,",
  "      dropout     = if (lstm_layers > 1) 0.2 else 0",
  "    )",
  "    self$head <- torch::nn_linear(hidden_dim, 1)",
  "  },",
  "  forward = function(x) {",
  "    b <- x$shape[1]; tt <- x$shape[2]; cc <- x$shape[3]",
  "    h <- x$shape[4]; w  <- x$shape[5]",
  "",
  "    skip <- x[, tt, 1, , ]$unsqueeze(2)",
  "",
  "    seq <- x$permute(c(1, 4, 5, 2, 3))$contiguous()$view(c(b * h * w, tt, cc))",
  "",
  "    out   <- self$lstm(seq)[[1]][, tt, ]",
  "    delta <- self$head(out)$view(c(b, 1, h, w))",
  "",
  "    skip + delta",
  "  }",
  ")",
  "",
  "HIDDEN_DIM  <- 32L",
  "LSTM_LAYERS <- 1L",
  "",
  "cat(sprintf(\"  hidden_dim: %d, lstm_layers: %d, epochs: %d, batch: %d\\n\",",
  "            HIDDEN_DIM, LSTM_LAYERS, EPOCHS, tp$batch_size))",
  "",
  "log_path <- here::here(\"output\", \"training_log_lstm.csv\")",
  "fitted_lstm <- load_or_train(",
  "  lstm_only,",
  "  hparams  = list(in_channels = meta$n_channels,",
  "                  hidden_dim  = HIDDEN_DIM,",
  "                  lstm_layers = LSTM_LAYERS),",
  "  dls            = dls,",
  "  log_path       = log_path,",
  "  model_filename = \"lstm_only_final.pt\",",
  "  lr             = tp$lr,",
  "  accelerator    = tp$accelerator,",
  "  fig_path       = here::here(\"output\", \"figures\", \"training_loss_lstm.png\"),",
  "  fig_title      = \"Per-cell LSTM — training loss\"",
  ")",
  "",
  "evaluate_and_save(",
  "  fitted_lstm, dls$test, tensors, meta,",
  "  results_path = here::here(\"data\", \"processed\", \"lstm_only_test_results.rds\"),",
  "  label        = \"Per-cell LSTM (skip)\"",
  ")"
))
files_written <- c(files_written, "code/06_lstm_only_skip.R")

write_code_file("code/07_lstm_only_noskip.R", c(
  "library(here)",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "",
  "cat(\"=== Per-cell LSTM (no skip, ablation) ===\\n\")",
  "cat(sprintf(\"  Grid: %d x %d, Window: %d, Channels: %d\\n\",",
  "            meta$grid_dim[[1]], meta$grid_dim[[2]],",
  "            meta$window_size, meta$n_channels))",
  "",
  "tp  <- arch_hparams(\"percell\")",
  "dls <- make_dataloaders(tensors, meta, batch_size = tp$batch_size)",
  "",
  "lstm_only_noskip <- torch::nn_module(",
  "  \"lstm_only_noskip\",",
  "  initialize = function(in_channels, hidden_dim, lstm_layers) {",
  "    self$lstm <- torch::nn_lstm(",
  "      input_size  = in_channels,",
  "      hidden_size = hidden_dim,",
  "      num_layers  = lstm_layers,",
  "      batch_first = TRUE,",
  "      dropout     = if (lstm_layers > 1) 0.2 else 0",
  "    )",
  "    self$head <- torch::nn_linear(hidden_dim, 1)",
  "  },",
  "  forward = function(x) {",
  "    b <- x$shape[1]; tt <- x$shape[2]; cc <- x$shape[3]",
  "    h <- x$shape[4]; w  <- x$shape[5]",
  "",
  "    seq <- x$permute(c(1, 4, 5, 2, 3))$contiguous()$view(c(b * h * w, tt, cc))",
  "",
  "    out <- self$lstm(seq)[[1]][, tt, ]",
  "    self$head(out)$view(c(b, 1, h, w))",
  "  }",
  ")",
  "",
  "HIDDEN_DIM  <- 32L",
  "LSTM_LAYERS <- 1L",
  "",
  "cat(sprintf(\"  hidden_dim: %d, lstm_layers: %d, epochs: %d, batch: %d\\n\",",
  "            HIDDEN_DIM, LSTM_LAYERS, EPOCHS, tp$batch_size))",
  "",
  "log_path <- here::here(\"output\", \"training_log_lstm_noskip.csv\")",
  "fitted_lstm_ns <- load_or_train(",
  "  lstm_only_noskip,",
  "  hparams  = list(in_channels = meta$n_channels,",
  "                  hidden_dim  = HIDDEN_DIM,",
  "                  lstm_layers = LSTM_LAYERS),",
  "  dls            = dls,",
  "  log_path       = log_path,",
  "  model_filename = \"lstm_only_noskip_final.pt\",",
  "  lr             = tp$lr,",
  "  accelerator    = tp$accelerator,",
  "  fig_path       = here::here(\"output\", \"figures\", \"training_loss_lstm_noskip.png\"),",
  "  fig_title      = \"Per-cell LSTM (no skip) — training loss\"",
  ")",
  "",
  "evaluate_and_save(",
  "  fitted_lstm_ns, dls$test, tensors, meta,",
  "  results_path = here::here(\"data\", \"processed\", \"lstm_only_noskip_test_results.rds\"),",
  "  label        = \"Per-cell LSTM (no skip)\"",
  ")"
))
files_written <- c(files_written, "code/07_lstm_only_noskip.R")

write_code_file("code/08_convlstm_skip.R", c(
  "library(here)",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "",
  "cat(\"=== ConvLSTM (spatially-aware recurrence) ===\\n\")",
  "cat(sprintf(\"  Grid: %d x %d, Window: %d, Channels: %d\\n\",",
  "            meta$grid_dim[[1]], meta$grid_dim[[2]],",
  "            meta$window_size, meta$n_channels))",
  "",
  "tp  <- arch_hparams(\"spatial\")",
  "dls <- make_dataloaders(tensors, meta, batch_size = tp$batch_size)",
  "",
  "convlstm_cell <- torch::nn_module(",
  "  \"convlstm_cell\",",
  "  initialize = function(in_channels, hidden_channels, kernel_size = 3) {",
  "    self$hidden_channels <- hidden_channels",
  "    pad <- kernel_size %/% 2",
  "    self$conv <- torch::nn_conv2d(",
  "      in_channels  = in_channels + hidden_channels,",
  "      out_channels = 4 * hidden_channels,",
  "      kernel_size  = kernel_size,",
  "      padding      = pad",
  "    )",
  "  },",
  "  forward = function(x, h, c) {",
  "    combined <- torch::torch_cat(list(x, h), dim = 2)",
  "    gates    <- self$conv(combined)",
  "    split    <- torch::torch_split(gates, self$hidden_channels, dim = 2)",
  "    i_t <- torch::torch_sigmoid(split[[1]])",
  "    f_t <- torch::torch_sigmoid(split[[2]])",
  "    g_t <- torch::torch_tanh(split[[3]])",
  "    o_t <- torch::torch_sigmoid(split[[4]])",
  "    c_next <- f_t * c + i_t * g_t",
  "    h_next <- o_t * torch::torch_tanh(c_next)",
  "    list(h_next, c_next)",
  "  }",
  ")",
  "",
  "convlstm_model <- torch::nn_module(",
  "  \"convlstm_model\",",
  "  initialize = function(in_channels, enc_channels, hidden_channels,",
  "                        grid_h, grid_w) {",
  "    self$encoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(in_channels, enc_channels, kernel_size = 3,",
  "                stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(enc_channels),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(enc_channels, enc_channels, kernel_size = 3,",
  "                stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(enc_channels),",
  "      torch::nn_relu()",
  "    )",
  "    self$cell <- convlstm_cell(enc_channels, hidden_channels, kernel_size = 3)",
  "",
  "    self$decoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(hidden_channels, enc_channels, kernel_size = 3, padding = 1),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(enc_channels, 1, kernel_size = 3, padding = 1)",
  "    )",
  "    self$upsample <- torch::nn_upsample(size = c(grid_h, grid_w), mode = \"bilinear\",",
  "                                 align_corners = FALSE)",
  "    self$hidden_channels <- hidden_channels",
  "  },",
  "  forward = function(x) {",
  "    b <- x$shape[1]; tt <- x$shape[2]",
  "    cc <- x$shape[3]; h <- x$shape[4]; w <- x$shape[5]",
  "",
  "    skip <- x[, tt, 1, , ]$unsqueeze(2)",
  "",
  "    x_flat <- x$reshape(c(b * tt, cc, h, w))",
  "    feat   <- self$encoder(x_flat)",
  "    feat   <- feat$view(c(b, tt, feat$shape[2],",
  "                          feat$shape[3], feat$shape[4]))",
  "",
  "    h_state <- torch::torch_zeros(b, self$hidden_channels, feat$shape[4], feat$shape[5],",
  "                           device = feat$device, dtype = feat$dtype)",
  "    c_state <- torch::torch_zeros_like(h_state)",
  "    for (t in seq_len(tt)) {",
  "      out <- self$cell(feat[, t, , , ], h_state, c_state)",
  "      h_state <- out[[1]]; c_state <- out[[2]]",
  "    }",
  "",
  "    delta <- self$decoder(h_state)",
  "    delta <- self$upsample(delta)",
  "    skip + delta",
  "  }",
  ")",
  "",
  "ENC_CHANNELS    <- 16L",
  "HIDDEN_CHANNELS <- 32L",
  "grid_h          <- as.integer(dim(tensors$source_array)[1])",
  "grid_w          <- as.integer(dim(tensors$source_array)[2])",
  "",
  "cat(sprintf(\"  enc_c: %d, hidden_c: %d, epochs: %d, batch: %d\\n\",",
  "            ENC_CHANNELS, HIDDEN_CHANNELS, EPOCHS, tp$batch_size))",
  "",
  "log_path <- here::here(\"output\", \"training_log_convlstm.csv\")",
  "fitted_cl <- load_or_train(",
  "  convlstm_model,",
  "  hparams  = list(in_channels     = meta$n_channels,",
  "                  enc_channels    = ENC_CHANNELS,",
  "                  hidden_channels = HIDDEN_CHANNELS,",
  "                  grid_h          = grid_h,",
  "                  grid_w          = grid_w),",
  "  dls            = dls,",
  "  log_path       = log_path,",
  "  model_filename = \"convlstm_final.pt\",",
  "  lr             = tp$lr,",
  "  accelerator    = tp$accelerator,",
  "  fig_path       = here::here(\"output\", \"figures\", \"training_loss_convlstm.png\"),",
  "  fig_title      = \"ConvLSTM — training loss\"",
  ")",
  "",
  "evaluate_and_save(",
  "  fitted_cl, dls$test, tensors, meta,",
  "  results_path = here::here(\"data\", \"processed\", \"convlstm_test_results.rds\"),",
  "  label        = \"ConvLSTM\"",
  ")"
))
files_written <- c(files_written, "code/08_convlstm_skip.R")

write_code_file("code/09_convlstm_noskip.R", c(
  "library(here)",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta    <- tensors$meta",
  "",
  "cat(\"=== ConvLSTM (no skip, ablation) ===\\n\")",
  "cat(sprintf(\"  Grid: %d x %d, Window: %d, Channels: %d\\n\",",
  "            meta$grid_dim[[1]], meta$grid_dim[[2]],",
  "            meta$window_size, meta$n_channels))",
  "",
  "tp  <- arch_hparams(\"spatial\")",
  "dls <- make_dataloaders(tensors, meta, batch_size = tp$batch_size)",
  "",
  "convlstm_cell <- torch::nn_module(",
  "  \"convlstm_cell\",",
  "  initialize = function(in_channels, hidden_channels, kernel_size = 3) {",
  "    self$hidden_channels <- hidden_channels",
  "    pad <- kernel_size %/% 2",
  "    self$conv <- torch::nn_conv2d(",
  "      in_channels  = in_channels + hidden_channels,",
  "      out_channels = 4 * hidden_channels,",
  "      kernel_size  = kernel_size,",
  "      padding      = pad",
  "    )",
  "  },",
  "  forward = function(x, h, c) {",
  "    combined <- torch::torch_cat(list(x, h), dim = 2)",
  "    gates    <- self$conv(combined)",
  "    split    <- torch::torch_split(gates, self$hidden_channels, dim = 2)",
  "    i_t <- torch::torch_sigmoid(split[[1]])",
  "    f_t <- torch::torch_sigmoid(split[[2]])",
  "    g_t <- torch::torch_tanh(split[[3]])",
  "    o_t <- torch::torch_sigmoid(split[[4]])",
  "    c_next <- f_t * c + i_t * g_t",
  "    h_next <- o_t * torch::torch_tanh(c_next)",
  "    list(h_next, c_next)",
  "  }",
  ")",
  "",
  "convlstm_model_noskip <- torch::nn_module(",
  "  \"convlstm_model_noskip\",",
  "  initialize = function(in_channels, enc_channels, hidden_channels,",
  "                        grid_h, grid_w) {",
  "    self$encoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(in_channels, enc_channels, kernel_size = 3,",
  "                stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(enc_channels),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(enc_channels, enc_channels, kernel_size = 3,",
  "                stride = 2, padding = 1),",
  "      torch::nn_batch_norm2d(enc_channels),",
  "      torch::nn_relu()",
  "    )",
  "    self$cell <- convlstm_cell(enc_channels, hidden_channels, kernel_size = 3)",
  "",
  "    self$decoder <- torch::nn_sequential(",
  "      torch::nn_conv2d(hidden_channels, enc_channels, kernel_size = 3, padding = 1),",
  "      torch::nn_relu(),",
  "      torch::nn_conv2d(enc_channels, 1, kernel_size = 3, padding = 1)",
  "    )",
  "    self$upsample <- torch::nn_upsample(size = c(grid_h, grid_w), mode = \"bilinear\",",
  "                                 align_corners = FALSE)",
  "    self$hidden_channels <- hidden_channels",
  "  },",
  "  forward = function(x) {",
  "    b <- x$shape[1]; tt <- x$shape[2]",
  "    cc <- x$shape[3]; h <- x$shape[4]; w <- x$shape[5]",
  "",
  "    x_flat <- x$reshape(c(b * tt, cc, h, w))",
  "    feat   <- self$encoder(x_flat)",
  "    feat   <- feat$view(c(b, tt, feat$shape[2],",
  "                          feat$shape[3], feat$shape[4]))",
  "",
  "    h_state <- torch::torch_zeros(b, self$hidden_channels, feat$shape[4], feat$shape[5],",
  "                           device = feat$device, dtype = feat$dtype)",
  "    c_state <- torch::torch_zeros_like(h_state)",
  "    for (t in seq_len(tt)) {",
  "      out <- self$cell(feat[, t, , , ], h_state, c_state)",
  "      h_state <- out[[1]]; c_state <- out[[2]]",
  "    }",
  "",
  "    y_hat <- self$decoder(h_state)",
  "    self$upsample(y_hat)",
  "  }",
  ")",
  "",
  "ENC_CHANNELS    <- 16L",
  "HIDDEN_CHANNELS <- 32L",
  "grid_h          <- as.integer(dim(tensors$source_array)[1])",
  "grid_w          <- as.integer(dim(tensors$source_array)[2])",
  "",
  "cat(sprintf(\"  enc_c: %d, hidden_c: %d, epochs: %d, batch: %d\\n\",",
  "            ENC_CHANNELS, HIDDEN_CHANNELS, EPOCHS, tp$batch_size))",
  "",
  "log_path <- here::here(\"output\", \"training_log_convlstm_noskip.csv\")",
  "fitted_cl_ns <- load_or_train(",
  "  convlstm_model_noskip,",
  "  hparams  = list(in_channels     = meta$n_channels,",
  "                  enc_channels    = ENC_CHANNELS,",
  "                  hidden_channels = HIDDEN_CHANNELS,",
  "                  grid_h          = grid_h,",
  "                  grid_w          = grid_w),",
  "  dls            = dls,",
  "  log_path       = log_path,",
  "  model_filename = \"convlstm_noskip_final.pt\",",
  "  lr             = tp$lr,",
  "  accelerator    = tp$accelerator,",
  "  fig_path       = here::here(\"output\", \"figures\", \"training_loss_convlstm_noskip.png\"),",
  "  fig_title      = \"ConvLSTM (no skip) — training loss\"",
  ")",
  "",
  "evaluate_and_save(",
  "  fitted_cl_ns, dls$test, tensors, meta,",
  "  results_path = here::here(\"data\", \"processed\", \"convlstm_noskip_test_results.rds\"),",
  "  label        = \"ConvLSTM (no skip)\"",
  ")"
))
files_written <- c(files_written, "code/09_convlstm_noskip.R")

write_code_file("code/10_evaluation.R", c(
  "library(tidyverse)",
  "library(stars)",
  "library(sf)",
  "library(spdep)",
  "library(rnaturalearth)",
  "library(rnaturalearthdata)",
  "library(here)",
  "",
  "source(here::here(\"code\", \"_common_training.R\"))",
  "",
  "GRID_NX <- 110L",
  "GRID_NY <- 60L",
  "LON_CENTERS <- 13.95 + seq_len(GRID_NX) * 0.1 - 0.05",
  "LAT_CENTERS <- 55.05 + seq_len(GRID_NY) * (-0.1) + 0.05",
  "",
  "borders_sf <- rnaturalearth::ne_countries(scale = \"medium\",",
  "                           country = c(\"Poland\", \"Germany\", \"Czech Republic\",",
  "                                       \"Slovakia\", \"Ukraine\", \"Belarus\",",
  "                                       \"Lithuania\", \"Russia\", \"Austria\"),",
  "                           returnclass = \"sf\")",
  "poland_sf <- borders_sf %>% dplyr::filter(admin == \"Poland\")",
  "",
  "grid_to_df <- function(mat, value_name = \"value\") {",
  "  df <- expand.grid(lon = LON_CENTERS, lat = LAT_CENTERS)",
  "  df[[value_name]] <- as.vector(mat)",
  "  df",
  "}",
  "",
  "cities_df <- tibble::tibble(",
  "  name = c(\"Warszawa\", \"Kraków\", \"Gdańsk\", \"Wrocław\", \"Poznań\", \"Katowice\"),",
  "  lon  = c(21.01, 19.94, 18.65, 17.04, 16.93, 19.03),",
  "  lat  = c(52.23, 50.06, 54.35, 51.11, 52.41, 50.26)",
  ")",
  "",
  "results  <- readRDS(here::here(\"data\", \"processed\", \"cnn_lstm_skip_test_results.rds\"))",
  "headline <- readRDS(here::here(\"data\", \"processed\", \"convlstm_test_results.rds\"))",
  "tensors  <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta     <- tensors$meta",
  "",
  "preds   <- headline$predictions",
  "actuals <- headline$actuals",
  "errors  <- preds - actuals",
  "",
  "cat(\"=== Evaluation ===\\n\")",
  "cat(sprintf(\"  Test samples: %d\\n\", dim(preds)[1]))",
  "cat(sprintf(\"  Headline model (ConvLSTM, skip): RMSE %.2f µg/m³, MAE %.2f µg/m³\\n\",",
  "            headline$rmse, headline$mae))",
  "cat(sprintf(\"  CNN-LSTM (skip) reference row:   RMSE %.2f µg/m³, MAE %.2f µg/m³\\n\",",
  "            results$rmse, results$mae))",
  "",
  "out_fig <- here::here(\"output\", \"figures\")",
  "out_tab <- here::here(\"output\", \"tables\")",
  "if (!dir.exists(out_fig)) dir.create(out_fig, recursive = TRUE)",
  "if (!dir.exists(out_tab)) dir.create(out_tab, recursive = TRUE)",
  "",
  "inv_scale   <- make_inv_scale(meta)",
  "persistence <- persistence_pm25(tensors, meta, inv_scale)",
  "",
  "rmse_persist <- sqrt(mean((persistence - actuals)^2, na.rm = TRUE))",
  "mae_persist  <- mean(abs(persistence - actuals), na.rm = TRUE)",
  "",
  "Y_train_pm25 <- inv_scale(tensors$Y_train)",
  "climatology_cell <- apply(Y_train_pm25, c(2, 3), mean, na.rm = TRUE)",
  "climatology_grid <- array(rep(as.vector(climatology_cell), times = dim(actuals)[1]),",
  "                          dim = c(dim(actuals)[2], dim(actuals)[3], dim(actuals)[1]))",
  "climatology_grid <- aperm(climatology_grid, c(3, 1, 2))",
  "rmse_spatial_mean <- sqrt(mean((climatology_grid - actuals)^2, na.rm = TRUE))",
  "mae_spatial_mean  <- mean(abs(climatology_grid - actuals), na.rm = TRUE)",
  "",
  "bl_file         <- here::here(\"data\", \"processed\", \"baselines_results.rds\")",
  "noskip_file     <- here::here(\"data\", \"processed\", \"noskip_test_results.rds\")",
  "lstm_file       <- here::here(\"data\", \"processed\", \"lstm_only_test_results.rds\")",
  "lstm_ns_file    <- here::here(\"data\", \"processed\", \"lstm_only_noskip_test_results.rds\")",
  "convlstm_file   <- here::here(\"data\", \"processed\", \"convlstm_test_results.rds\")",
  "convlstm_ns_file <- here::here(\"data\", \"processed\", \"convlstm_noskip_test_results.rds\")",
  "baselines   <- if (file.exists(bl_file))          readRDS(bl_file)          else NULL",
  "noskip      <- if (file.exists(noskip_file))      readRDS(noskip_file)      else NULL",
  "lstm_only   <- if (file.exists(lstm_file))        readRDS(lstm_file)        else NULL",
  "lstm_only_ns <- if (file.exists(lstm_ns_file))    readRDS(lstm_ns_file)     else NULL",
  "convlstm    <- if (file.exists(convlstm_file))    readRDS(convlstm_file)    else NULL",
  "convlstm_ns <- if (file.exists(convlstm_ns_file)) readRDS(convlstm_ns_file) else NULL",
  "",
  "acc_rows <- list(",
  "  tibble::tibble(Model = \"CNN-LSTM (skip)\",     RMSE = results$rmse, MAE = results$mae)",
  ")",
  "if (!is.null(convlstm)) acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"ConvLSTM (skip)\",     RMSE = convlstm$rmse,   MAE = convlstm$mae)))",
  "if (!is.null(convlstm_ns)) acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"ConvLSTM (no skip)\",  RMSE = convlstm_ns$rmse, MAE = convlstm_ns$mae)))",
  "if (!is.null(lstm_only)) acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"Per-cell LSTM (skip)\", RMSE = lstm_only$rmse, MAE = lstm_only$mae)))",
  "if (!is.null(lstm_only_ns)) acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"Per-cell LSTM (no skip)\", RMSE = lstm_only_ns$rmse, MAE = lstm_only_ns$mae)))",
  "if (!is.null(noskip)) acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"CNN-LSTM (no skip)\",  RMSE = noskip$rmse,          MAE = noskip$mae)))",
  "if (!is.null(baselines)) acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"STAR (spatial+lag)\",  RMSE = baselines$star$rmse,  MAE = baselines$star$mae),",
  "  tibble::tibble(Model = \"AR(1) per cell\",      RMSE = baselines$ar1$rmse,   MAE = baselines$ar1$mae)))",
  "acc_rows <- c(acc_rows, list(",
  "  tibble::tibble(Model = \"Persistence (naive)\",            RMSE = rmse_persist,          MAE = mae_persist),",
  "  tibble::tibble(Model = \"Training climatology (per cell)\", RMSE = rmse_spatial_mean,    MAE = mae_spatial_mean)))",
  "",
  "accuracy <- dplyr::bind_rows(acc_rows) %>%",
  "  dplyr::mutate(`RMSE vs persistence` = sprintf(\"%+.1f%%\", (1 - RMSE / rmse_persist) * 100))",
  "",
  "cat(\"\\n=== Accuracy Comparison ===\\n\")",
  "print(accuracy)",
  "readr::write_csv(accuracy, file.path(out_tab, \"accuracy_comparison.csv\"))",
  "",
  "exceedance_metrics <- function(preds_arr, actuals_arr, thr = 15) {",
  "  a <- actuals_arr > thr; p <- preds_arr > thr",
  "  tp <- sum(p &  a, na.rm = TRUE)",
  "  fp <- sum(p & !a, na.rm = TRUE)",
  "  fn <- sum(!p & a, na.rm = TRUE)",
  "  tibble::tibble(actual_rate    = mean(a, na.rm = TRUE),",
  "         predicted_rate = mean(p, na.rm = TRUE),",
  "         recall         = if ((tp + fn) == 0) NA_real_ else tp / (tp + fn),",
  "         precision      = if ((tp + fp) == 0) NA_real_ else tp / (tp + fp))",
  "}",
  "",
  "exc_rows <- list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"CNN-LSTM (skip)\"),",
  "            exceedance_metrics(results$predictions, actuals))",
  ")",
  "if (!is.null(convlstm)) exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"ConvLSTM (skip)\"),",
  "            exceedance_metrics(convlstm$predictions, actuals))))",
  "if (!is.null(convlstm_ns)) exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"ConvLSTM (no skip)\"),",
  "            exceedance_metrics(convlstm_ns$predictions, actuals))))",
  "if (!is.null(lstm_only)) exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"Per-cell LSTM (skip)\"),",
  "            exceedance_metrics(lstm_only$predictions, actuals))))",
  "if (!is.null(lstm_only_ns)) exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"Per-cell LSTM (no skip)\"),",
  "            exceedance_metrics(lstm_only_ns$predictions, actuals))))",
  "if (!is.null(noskip)) exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"CNN-LSTM (no skip)\"),",
  "            exceedance_metrics(noskip$predictions, actuals))))",
  "if (!is.null(baselines)) exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"STAR (spatial+lag)\"),",
  "            exceedance_metrics(baselines$star$predictions, actuals)),",
  "  dplyr::bind_cols(tibble::tibble(Model = \"AR(1) per cell\"),",
  "            exceedance_metrics(baselines$ar1$predictions, actuals))))",
  "exc_rows <- c(exc_rows, list(",
  "  dplyr::bind_cols(tibble::tibble(Model = \"Persistence\"),",
  "            exceedance_metrics(persistence, actuals))))",
  "",
  "exceedance <- dplyr::bind_rows(exc_rows)",
  "cat(\"\\n=== Exceedance skill (>15 µg/m³, WHO 2021 AQG 24h, daily cells) ===\\n\")",
  "print(exceedance)",
  "readr::write_csv(exceedance, file.path(out_tab, \"exceedance_skill.csv\"))",
  "",
  "rmse_daily <- sapply(1:dim(preds)[1], function(i) {",
  "  sqrt(mean((preds[i, , ] - actuals[i, , ])^2, na.rm = TRUE))",
  "})",
  "",
  "test_times_samples <- meta$times[meta$split_idx$test + meta$window_size]",
  "",
  "p_rmse_ts <- ggplot2::ggplot(",
  "  tibble::tibble(time = test_times_samples, rmse = rmse_daily),",
  "  ggplot2::aes(x = time, y = rmse)",
  ") +",
  "  ggplot2::geom_line(color = \"steelblue\", alpha = 0.7) +",
  "  ggplot2::geom_smooth(method = \"loess\", span = 0.2, se = FALSE, color = \"tomato\") +",
  "  ggplot2::labs(title = \"ConvLSTM (skip) Forecast Error Over Time (Test Set)\",",
  "       y = expression(RMSE~\"[\"*mu*g/m^3*\"]\"),",
  "       x = NULL) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"test_rmse_timeseries.png\"), p_rmse_ts,",
  "       width = 10, height = 5)",
  "",
  "mae_grid <- headline$mae_grid",
  "bias_grid <- apply(errors, c(2, 3), mean, na.rm = TRUE)",
  "",
  "mae_df <- grid_to_df(mae_grid, \"mae\")",
  "",
  "p_mae_spatial <- ggplot2::ggplot() +",
  "  ggplot2::geom_raster(data = mae_df, ggplot2::aes(lon, lat, fill = mae)) +",
  "  ggplot2::geom_sf(data = borders_sf, fill = NA, colour = \"grey25\", linewidth = 0.3) +",
  "  ggplot2::geom_sf(data = poland_sf, fill = NA, colour = \"black\", linewidth = 0.6) +",
  "  ggplot2::geom_point(data = cities_df, ggplot2::aes(lon, lat), colour = \"white\", size = 1.6) +",
  "  ggplot2::geom_text(data = cities_df, ggplot2::aes(lon, lat, label = name),",
  "            colour = \"white\", size = 3, nudge_y = 0.12) +",
  "  ggplot2::scale_fill_viridis_c(name = expression(MAE~\"[\"*mu*g/m^3*\"]\"),",
  "                       option = \"inferno\") +",
  "  ggplot2::coord_sf(xlim = range(LON_CENTERS), ylim = range(LAT_CENTERS), expand = FALSE) +",
  "  ggplot2::labs(title = \"Spatial distribution of ConvLSTM (skip) forecast error (MAE)\",",
  "       subtitle = \"Headline model, averaged across the 270-day test set\",",
  "       x = NULL, y = NULL) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"test_mae_spatial.png\"), p_mae_spatial,",
  "       width = 8, height = 6, dpi = 150)",
  "",
  "example_idx <- which.max(rmse_daily)",
  "example_date <- test_times_samples[example_idx]",
  "",
  "shared_limits <- range(c(actuals[example_idx, , ], preds[example_idx, , ]),",
  "                       na.rm = TRUE)",
  "",
  "map_layer <- function(df, value_col, title, scale_type = c(\"viridis\", \"diverging\")) {",
  "  scale_type <- match.arg(scale_type)",
  "  g <- ggplot2::ggplot() +",
  "    ggplot2::geom_raster(data = df, ggplot2::aes(lon, lat, fill = .data[[value_col]])) +",
  "    ggplot2::geom_sf(data = borders_sf, fill = NA, colour = \"grey30\", linewidth = 0.3) +",
  "    ggplot2::geom_sf(data = poland_sf, fill = NA, colour = \"black\", linewidth = 0.6) +",
  "    ggplot2::coord_sf(xlim = range(LON_CENTERS), ylim = range(LAT_CENTERS), expand = FALSE) +",
  "    ggplot2::labs(title = title, x = NULL, y = NULL) +",
  "    ggplot2::theme_minimal() +",
  "    ggplot2::theme(legend.position = \"bottom\")",
  "  if (scale_type == \"viridis\") {",
  "    g + ggplot2::scale_fill_viridis_c(name = expression(mu*g/m^3),",
  "                             limits = shared_limits)",
  "  } else {",
  "    g + ggplot2::scale_fill_gradient2(name = expression(mu*g/m^3),",
  "                             low = \"#2166ac\", mid = \"white\", high = \"#b2182b\",",
  "                             midpoint = 0)",
  "  }",
  "}",
  "",
  "p_actual <- map_layer(grid_to_df(actuals[example_idx, , ], \"v\"), \"v\",",
  "                      sprintf(\"Actual — %s\", example_date))",
  "p_pred   <- map_layer(grid_to_df(preds[example_idx, , ], \"v\"), \"v\",",
  "                      sprintf(\"Predicted — %s\", example_date))",
  "p_err    <- map_layer(grid_to_df(errors[example_idx, , ], \"v\"), \"v\",",
  "                      sprintf(\"Error (pred − actual)\"),",
  "                      scale_type = \"diverging\")",
  "",
  "p_example <- cowplot::plot_grid(p_actual, p_pred, p_err, ncol = 3)",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"test_example_prediction.png\"), p_example,",
  "       width = 15, height = 6, dpi = 150)",
  "",
  "nx <- dim(errors)[2]",
  "ny <- dim(errors)[3]",
  "coords <- expand.grid(lon = LON_CENTERS, lat = LAT_CENTERS)",
  "knn <- spdep::knearneigh(as.matrix(coords), k = 8)",
  "nb  <- spdep::knn2nb(knn)",
  "lw  <- spdep::nb2listw(nb, style = \"W\")",
  "",
  "mean_errors <- apply(errors, c(2, 3), mean, na.rm = TRUE)",
  "error_vec   <- as.vector(mean_errors)",
  "valid       <- !is.na(error_vec)",
  "coords_valid <- coords[valid, ]",
  "error_valid  <- error_vec[valid]",
  "",
  "if (sum(valid) < length(error_vec)) {",
  "  knn_v <- spdep::knearneigh(as.matrix(coords_valid), k = 8)",
  "  lw_mean <- spdep::nb2listw(spdep::knn2nb(knn_v), style = \"W\")",
  "} else {",
  "  lw_mean <- lw",
  "}",
  "moran_mean <- spdep::moran.test(error_valid, lw_mean)",
  "",
  "n_test_days  <- dim(errors)[1]",
  "moran_per_day <- numeric(n_test_days)",
  "for (d in seq_len(n_test_days)) {",
  "  err_day <- as.vector(errors[d, , ])",
  "  valid_d <- !is.na(err_day)",
  "  if (sum(valid_d) < 10) { moran_per_day[d] <- NA_real_; next }",
  "  if (sum(valid_d) < length(err_day)) {",
  "    knn_d <- spdep::knearneigh(as.matrix(coords[valid_d, ]),",
  "                        k = min(8, sum(valid_d) - 1))",
  "    lw_d  <- spdep::nb2listw(spdep::knn2nb(knn_d), style = \"W\")",
  "    moran_per_day[d] <- spdep::moran.test(err_day[valid_d], lw_d)$estimate[\"Moran I statistic\"]",
  "  } else {",
  "    moran_per_day[d] <- spdep::moran.test(err_day, lw)$estimate[\"Moran I statistic\"]",
  "  }",
  "}",
  "",
  "per_day_q <- quantile(moran_per_day, c(0.25, 0.5, 0.75), na.rm = TRUE)",
  "",
  "cat(\"\\n=== Moran's I on Forecast Errors ===\\n\")",
  "cat(sprintf(\"  (a) Mean residual field (persistent bias):\\n\"))",
  "cat(sprintf(\"      I = %.4f, p = %.6f\\n\",",
  "            moran_mean$estimate[\"Moran I statistic\"], moran_mean$p.value))",
  "cat(sprintf(\"  (b) Per-day distribution (transient clustering), n = %d days:\\n\",",
  "            n_test_days))",
  "cat(sprintf(\"      median = %.4f, IQR = [%.4f, %.4f], range = [%.4f, %.4f]\\n\",",
  "            per_day_q[2], per_day_q[1], per_day_q[3],",
  "            min(moran_per_day, na.rm = TRUE),",
  "            max(moran_per_day, na.rm = TRUE)))",
  "if (moran_mean$p.value < 0.05) {",
  "  cat(\"  -> Persistent spatial bias: same cells are systematically wrong.\\n\")",
  "} else {",
  "  cat(\"  -> No persistent bias detected on the mean field.\\n\")",
  "}",
  "",
  "moran_summary <- tibble::tibble(",
  "  source    = c(\"mean_residual_field\",",
  "                \"per_day_median\", \"per_day_iqr_low\", \"per_day_iqr_high\",",
  "                \"per_day_min\", \"per_day_max\"),",
  "  statistic = c(as.numeric(moran_mean$estimate[\"Moran I statistic\"]),",
  "                per_day_q[2], per_day_q[1], per_day_q[3],",
  "                min(moran_per_day, na.rm = TRUE),",
  "                max(moran_per_day, na.rm = TRUE)),",
  "  p_value   = c(moran_mean$p.value, rep(NA_real_, 5))",
  ")",
  "readr::write_csv(moran_summary, file.path(out_tab, \"moran_i_residuals.csv\"))",
  "",
  "lisa <- spdep::localmoran(error_valid, lw_mean)",
  "",
  "coords_valid$lisa_Ii    <- lisa[, \"Ii\"]",
  "coords_valid$lisa_pval  <- lisa[, \"Pr(z != E(Ii))\"]",
  "coords_valid$mean_error <- error_valid",
  "",
  "lag_error <- spdep::lag.listw(lw_mean, error_valid)",
  "coords_valid$quadrant <- dplyr::case_when(",
  "  error_valid > 0 & lag_error > 0 ~ \"High-High\",",
  "  error_valid < 0 & lag_error < 0 ~ \"Low-Low\",",
  "  error_valid > 0 & lag_error < 0 ~ \"High-Low\",",
  "  error_valid < 0 & lag_error > 0 ~ \"Low-High\"",
  ")",
  "coords_valid$quadrant[coords_valid$lisa_pval > 0.05] <- \"Not significant\"",
  "",
  "p_lisa <- ggplot2::ggplot() +",
  "  ggplot2::geom_raster(data = coords_valid, ggplot2::aes(lon, lat, fill = quadrant)) +",
  "  ggplot2::geom_sf(data = borders_sf, fill = NA, colour = \"grey25\", linewidth = 0.3) +",
  "  ggplot2::geom_sf(data = poland_sf, fill = NA, colour = \"black\", linewidth = 0.6) +",
  "  ggplot2::geom_point(data = cities_df, ggplot2::aes(lon, lat), colour = \"black\", size = 1.3) +",
  "  ggplot2::geom_text(data = cities_df, ggplot2::aes(lon, lat, label = name),",
  "            colour = \"black\", size = 3, nudge_y = 0.15, fontface = \"bold\") +",
  "  ggplot2::scale_fill_manual(values = c(",
  "    \"High-High\"       = \"#d73027\",",
  "    \"Low-Low\"         = \"#4575b4\",",
  "    \"High-Low\"        = \"#fdae61\",",
  "    \"Low-High\"        = \"#abd9e9\",",
  "    \"Not significant\" = \"grey90\"",
  "  )) +",
  "  ggplot2::coord_sf(xlim = range(LON_CENTERS), ylim = range(LAT_CENTERS), expand = FALSE) +",
  "  ggplot2::labs(title = \"LISA cluster map of ConvLSTM (skip) forecast errors\",",
  "       subtitle = \"Where does the headline model systematically over- or under-predict?\",",
  "       fill = \"Cluster\", x = NULL, y = NULL) +",
  "  ggplot2::theme_minimal()",
  "",
  "ggplot2::ggsave(file.path(out_fig, \"lisa_error_clusters.png\"), p_lisa,",
  "       width = 8, height = 6, dpi = 150)",
  "",
  "arch_models <- list()",
  "arch_models[[\"CNN-LSTM (skip)\"]] <- results$mae_grid",
  "if (!is.null(noskip))       arch_models[[\"CNN-LSTM (no skip)\"]]      <- noskip$mae_grid",
  "if (!is.null(lstm_only))    arch_models[[\"Per-cell LSTM (skip)\"]]    <- lstm_only$mae_grid",
  "if (!is.null(lstm_only_ns)) arch_models[[\"Per-cell LSTM (no skip)\"]] <- lstm_only_ns$mae_grid",
  "if (!is.null(convlstm))     arch_models[[\"ConvLSTM (skip)\"]]         <- convlstm$mae_grid",
  "if (!is.null(convlstm_ns))  arch_models[[\"ConvLSTM (no skip)\"]]      <- convlstm_ns$mae_grid",
  "",
  "if (length(arch_models) >= 2) {",
  "  arch_df <- purrr::imap_dfr(arch_models, ~ {",
  "    df <- grid_to_df(.x, \"mae\"); df$Model <- .y; df",
  "  })",
  "  arch_df$Model <- factor(arch_df$Model, levels = names(arch_models))",
  "",
  "  mae_cap <- quantile(arch_df$mae, 0.99, na.rm = TRUE)",
  "  p_arch <- ggplot2::ggplot() +",
  "    ggplot2::geom_raster(data = arch_df, ggplot2::aes(lon, lat, fill = pmin(mae, mae_cap))) +",
  "    ggplot2::geom_sf(data = borders_sf, fill = NA, colour = \"grey25\", linewidth = 0.3) +",
  "    ggplot2::geom_sf(data = poland_sf, fill = NA, colour = \"black\", linewidth = 0.6) +",
  "    ggplot2::scale_fill_viridis_c(name = expression(MAE~\"[\"*mu*g/m^3*\"]\"),",
  "                         option = \"inferno\") +",
  "    ggplot2::coord_sf(xlim = range(LON_CENTERS), ylim = range(LAT_CENTERS),",
  "             expand = FALSE) +",
  "    ggplot2::labs(title = \"Spatial MAE by architecture\",",
  "         subtitle = \"Where does spatial context help? Where does it fail?\",",
  "         x = NULL, y = NULL) +",
  "    ggplot2::facet_wrap(~ Model, ncol = 2) +",
  "    ggplot2::theme_minimal() +",
  "    ggplot2::theme(legend.position = \"bottom\")",
  "  ggplot2::ggsave(file.path(out_fig, \"arch_comparison_mae_maps.png\"), p_arch,",
  "         width = 11, height = 8, dpi = 150)",
  "}",
  "",
  "if (!is.null(arch_models[[\"ConvLSTM (skip)\"]])) {",
  "  nearest_idx <- function(target, centers) which.min(abs(centers - target))",
  "  ranking_cities <- tibble::tibble(",
  "    name = c(\"Warszawa\", \"Kraków\", \"Łódź\", \"Wrocław\", \"Poznań\", \"Gdańsk\",",
  "             \"Szczecin\", \"Bydgoszcz\", \"Lublin\", \"Katowice\", \"Białystok\", \"Rzeszów\"),",
  "    lon  = c(21.01, 19.94, 19.46, 17.04, 16.93, 18.65,",
  "             14.55, 18.00, 22.57, 19.03, 23.17, 22.00),",
  "    lat  = c(52.23, 50.06, 51.76, 51.11, 52.41, 54.35,",
  "             53.43, 53.12, 51.25, 50.26, 53.13, 50.04)",
  "  )",
  "  headline_mae_grid <- arch_models[[\"ConvLSTM (skip)\"]]",
  "  city_mae <- ranking_cities %>%",
  "    dplyr::mutate(MAE = mapply(function(lo, la)",
  "                     headline_mae_grid[nearest_idx(lo, LON_CENTERS),",
  "                                       nearest_idx(la, LAT_CENTERS)],",
  "                   lon, lat)) %>%",
  "    dplyr::arrange(MAE) %>%",
  "    dplyr::mutate(name = factor(name, levels = name))",
  "  p_city <- ggplot2::ggplot(city_mae, ggplot2::aes(MAE, name)) +",
  "    ggplot2::geom_col(fill = \"steelblue\") +",
  "    ggplot2::labs(title = \"Per-city test MAE — ConvLSTM (skip)\",",
  "         x = expression(MAE~\"[\"*mu*g/m^3*\"]\"), y = NULL) +",
  "    ggplot2::theme_minimal()",
  "  ggplot2::ggsave(file.path(out_fig, \"arch_comparison_mae_cities.png\"), p_city,",
  "         width = 8, height = 6, dpi = 150)",
  "}",
  "",
  "LR_STEP_SIZE <- 15",
  "training_logs <- list(",
  "  \"CNN-LSTM (skip)\"         = \"training_log_cnnlstm.csv\",",
  "  \"CNN-LSTM (no skip)\"      = \"training_log_cnnlstm_noskip.csv\",",
  "  \"Per-cell LSTM (skip)\"    = \"training_log_lstm.csv\",",
  "  \"Per-cell LSTM (no skip)\" = \"training_log_lstm_noskip.csv\",",
  "  \"ConvLSTM (skip)\"         = \"training_log_convlstm.csv\",",
  "  \"ConvLSTM (no skip)\"      = \"training_log_convlstm_noskip.csv\"",
  ")",
  "",
  "log_rows <- list(); stop_pts <- list(); best_pts <- list(); decay_pts <- list()",
  "for (m in names(training_logs)) {",
  "  fp <- here::here(\"output\", training_logs[[m]])",
  "  if (!file.exists(fp)) next",
  "  tl <- readr::read_csv(fp, show_col_types = FALSE)",
  "  tl$model <- m",
  "  log_rows[[m]] <- tl",
  "",
  "  stop_pts[[m]] <- tibble::tibble(model = m, epoch = max(tl$epoch),",
  "                          loss = tl$loss[tl$epoch == max(tl$epoch) &",
  "                                          tl$set == \"valid\"][1])",
  "",
  "  val_rows <- dplyr::filter(tl, set == \"valid\")",
  "  best_row <- val_rows[which.min(val_rows$loss), ]",
  "  best_pts[[m]] <- tibble::tibble(model = m, epoch = best_row$epoch, loss = best_row$loss)",
  "",
  "  max_epoch <- max(tl$epoch)",
  "  decays <- if (max_epoch >= LR_STEP_SIZE) {",
  "    seq(LR_STEP_SIZE, max_epoch, by = LR_STEP_SIZE)",
  "  } else integer(0)",
  "  if (length(decays)) decay_pts[[m]] <- tibble::tibble(model = m, epoch = decays)",
  "}",
  "",
  "if (length(log_rows)) {",
  "  tl_all    <- dplyr::bind_rows(log_rows)",
  "  stop_all  <- dplyr::bind_rows(stop_pts)",
  "  best_all  <- dplyr::bind_rows(best_pts)",
  "  decay_all <- if (length(decay_pts)) dplyr::bind_rows(decay_pts)",
  "               else tibble::tibble(model = character(), epoch = integer())",
  "",
  "  p_curves <- ggplot2::ggplot(tl_all, ggplot2::aes(epoch, loss, color = set)) +",
  "    ggplot2::geom_vline(data = decay_all, ggplot2::aes(xintercept = epoch),",
  "               linetype = \"dashed\", colour = \"grey60\", linewidth = 0.3) +",
  "    ggplot2::geom_line(linewidth = 0.6) +",
  "    ggplot2::geom_point(data = stop_all, ggplot2::aes(epoch, loss),",
  "               inherit.aes = FALSE, shape = 4, size = 2.5, stroke = 1) +",
  "    ggplot2::geom_point(data = best_all, ggplot2::aes(epoch, loss),",
  "               inherit.aes = FALSE, shape = 21, size = 2.5, stroke = 1,",
  "               fill = \"gold\", colour = \"black\") +",
  "    ggplot2::facet_wrap(~ model, scales = \"free\", ncol = 2) +",
  "    ggplot2::scale_color_manual(values = c(train = \"steelblue\", valid = \"tomato\"),",
  "                       labels = c(train = \"Train\", valid = \"Validation\")) +",
  "    ggplot2::labs(title = \"Training curves (MSE loss) with LR-decay, best-epoch, early-stop markers\",",
  "         x = \"Epoch\", y = \"MSE loss\", color = NULL,",
  "         caption = paste(",
  "           \"Dashed lines: learning-rate decay steps.\",",
  "           \"Gold circle: best validation epoch (weights restored via keep_best_model — test metrics reflect these).\",",
  "           \"Cross: early-stop epoch.\",",
  "           sep = \"  \")) +",
  "    ggplot2::theme_minimal(base_size = 10) +",
  "    ggplot2::theme(legend.position = \"top\")",
  "",
  "  ggplot2::ggsave(file.path(out_fig, \"training_curves_all_models.png\"), p_curves,",
  "         width = 9, height = 7, dpi = 150)",
  "}",
  "",
  "cat(\"\\n=== Figures saved ===\\n\")",
  "cat(\"  output/figures/test_rmse_timeseries.png\\n\")",
  "cat(\"  output/figures/test_example_prediction.png\\n\")",
  "cat(\"  output/figures/test_mae_spatial.png\\n\")",
  "cat(\"  output/figures/lisa_error_clusters.png\\n\")",
  "cat(\"  output/figures/arch_comparison_mae_maps.png\\n\")",
  "cat(\"  output/figures/arch_comparison_mae_cities.png\\n\")",
  "cat(\"  output/figures/training_curves_all_models.png\\n\")",
  "cat(\"\\n=== Tables saved ===\\n\")",
  "cat(\"  output/tables/accuracy_comparison.csv\\n\")",
  "cat(\"  output/tables/moran_i_residuals.csv\\n\")"
))
files_written <- c(files_written, "code/10_evaluation.R")

write_code_file("code/11_city_forecasts.R", c(
  "library(tidyverse)",
  "library(here)",
  "library(plotly)",
  "library(htmltools)",
  "library(htmlwidgets)",
  "",
  "nearest_idx <- function(target, centers) which.min(abs(centers - target))",
  "",
  "cities_df <- tibble::tibble(",
  "  name = c(",
  "    \"Warszawa\", \"Kraków\", \"Łódź\", \"Wrocław\", \"Poznań\", \"Gdańsk\",",
  "    \"Szczecin\", \"Bydgoszcz\", \"Lublin\", \"Katowice\", \"Białystok\", \"Rzeszów\"",
  "  ),",
  "  lon = c(",
  "    21.01, 19.94, 19.46, 17.04, 16.93, 18.65,",
  "    14.55, 18.00, 22.57, 19.03, 23.17, 22.00",
  "  ),",
  "  lat = c(",
  "    52.23, 50.06, 51.76, 51.11, 52.41, 54.35,",
  "    53.43, 53.12, 51.25, 50.26, 53.13, 50.04",
  "  )",
  ")",
  "",
  "tensors <- readRDS(here::here(\"data\", \"processed\", \"pm25_tensors.rds\"))",
  "meta <- tensors$meta",
  "test_dates <- meta$times[meta$split_idx$test + meta$window_size]",
  "",
  "GRID_NX     <- as.integer(dim(tensors$source_array)[1])",
  "GRID_NY     <- as.integer(dim(tensors$source_array)[2])",
  "LON_CENTERS <- 13.95 + seq_len(GRID_NX) * 0.1 - 0.05",
  "LAT_CENTERS <- 55.05 + seq_len(GRID_NY) * (-0.1) + 0.05",
  "",
  "inv_scale <- function(x) {",
  "  unscaled <- x * (meta$pm25_max - meta$pm25_min) + meta$pm25_min",
  "  if (isTRUE(meta$log_transform)) expm1(unscaled) else unscaled",
  "}",
  "",
  "read_if <- function(f) if (file.exists(f)) readRDS(f) else NULL",
  "cnn           <- read_if(here::here(\"data\", \"processed\", \"cnn_lstm_skip_test_results.rds\"))",
  "noskip        <- read_if(here::here(\"data\", \"processed\", \"noskip_test_results.rds\"))",
  "lstm_one      <- read_if(here::here(\"data\", \"processed\", \"lstm_only_test_results.rds\"))",
  "lstm_one_ns   <- read_if(here::here(\"data\", \"processed\", \"lstm_only_noskip_test_results.rds\"))",
  "convlstm      <- read_if(here::here(\"data\", \"processed\", \"convlstm_test_results.rds\"))",
  "convlstm_ns   <- read_if(here::here(\"data\", \"processed\", \"convlstm_noskip_test_results.rds\"))",
  "basel         <- read_if(here::here(\"data\", \"processed\", \"baselines_results.rds\"))",
  "",
  "actuals <- if (!is.null(cnn)) cnn$actuals else inv_scale(tensors$Y_test)",
  "",
  "src <- tensors$source_array",
  "persistence <- array(dim = c(length(meta$split_idx$test), GRID_NX, GRID_NY))",
  "for (i in seq_along(meta$split_idx$test)) {",
  "  t_last <- meta$split_idx$test[i] + meta$window_size - 1",
  "  persistence[i, , ] <- src[, , t_last, 1]",
  "}",
  "persistence <- inv_scale(persistence)",
  "",
  "model_preds <- list()",
  "if (!is.null(cnn))         model_preds[[\"CNN-LSTM (skip)\"]]         <- cnn$predictions",
  "if (!is.null(noskip))      model_preds[[\"CNN-LSTM (no skip)\"]]      <- noskip$predictions",
  "if (!is.null(lstm_one))    model_preds[[\"Per-cell LSTM (skip)\"]]    <- lstm_one$predictions",
  "if (!is.null(lstm_one_ns)) model_preds[[\"Per-cell LSTM (no skip)\"]] <- lstm_one_ns$predictions",
  "if (!is.null(convlstm))    model_preds[[\"ConvLSTM (skip)\"]]         <- convlstm$predictions",
  "if (!is.null(convlstm_ns)) model_preds[[\"ConvLSTM (no skip)\"]]      <- convlstm_ns$predictions",
  "if (!is.null(basel))       model_preds[[\"AR(1) per cell\"]]          <- basel$ar1$predictions",
  "if (!is.null(basel))       model_preds[[\"STAR\"]]                    <- basel$star$predictions",
  "model_preds[[\"Persistence\"]] <- persistence",
  "",
  "model_colors <- c(",
  "  \"Actual\"                  = \"#000000\",",
  "  \"CNN-LSTM (skip)\"         = \"#1f77b4\",",
  "  \"CNN-LSTM (no skip)\"      = \"#9467bd\",",
  "  \"Per-cell LSTM (skip)\"    = \"#ff7f0e\",",
  "  \"Per-cell LSTM (no skip)\" = \"#d62728\",",
  "  \"ConvLSTM (skip)\"         = \"#2ca02c\",",
  "  \"ConvLSTM (no skip)\"      = \"#17becf\",",
  "  \"AR(1) per cell\"          = \"#8c564b\",",
  "  \"STAR\"                    = \"#e377c2\",",
  "  \"Persistence\"             = \"#7f7f7f\"",
  ")",
  "",
  "city_series <- purrr::pmap(cities_df, function(name, lon, lat) {",
  "  i <- nearest_idx(lon, LON_CENTERS)",
  "  j <- nearest_idx(lat, LAT_CENTERS)",
  "  rows <- list(tibble::tibble(",
  "    date = test_dates, model = \"Actual\",",
  "    pm25 = actuals[, i, j]",
  "  ))",
  "  for (mn in names(model_preds)) {",
  "    rows <- c(rows, list(tibble::tibble(",
  "      date = test_dates, model = mn,",
  "      pm25 = model_preds[[mn]][, i, j]",
  "    )))",
  "  }",
  "  dplyr::bind_rows(rows) %>% dplyr::mutate(",
  "    grid_lon = LON_CENTERS[i],",
  "    grid_lat = LAT_CENTERS[j]",
  "  )",
  "})",
  "names(city_series) <- cities_df$name",
  "",
  "city_metrics <- dplyr::bind_rows(lapply(cities_df$name, function(cn) {",
  "  df <- city_series[[cn]]",
  "  act <- df %>%",
  "    dplyr::filter(model == \"Actual\") %>%",
  "    dplyr::pull(pm25)",
  "  purrr::map_dfr(setdiff(unique(df$model), \"Actual\"), function(mn) {",
  "    ph <- df %>%",
  "      dplyr::filter(model == mn) %>%",
  "      dplyr::pull(pm25)",
  "    tibble::tibble(",
  "      City = cn, Model = mn,",
  "      RMSE = sqrt(mean((ph - act)^2, na.rm = TRUE)),",
  "      MAE = mean(abs(ph - act), na.rm = TRUE)",
  "    )",
  "  })",
  "}))",
  "out_tab <- here::here(\"output\", \"tables\")",
  "if (!dir.exists(out_tab)) dir.create(out_tab, recursive = TRUE)",
  "readr::write_csv(city_metrics, file.path(out_tab, \"city_metrics.csv\"))",
  "",
  "build_city_plot <- function(city_df, cn) {",
  "  levs <- names(model_colors)",
  "  levs <- c(\"Actual\", setdiff(levs[levs %in% unique(city_df$model)], \"Actual\"))",
  "",
  "  p <- plotly::plot_ly(height = 560)",
  "  for (lv in levs) {",
  "    d <- city_df %>%",
  "      dplyr::filter(model == lv) %>%",
  "      dplyr::arrange(date)",
  "    p <- p %>% plotly::add_trace(",
  "      x = d$date, y = d$pm25,",
  "      name = lv,",
  "      type = \"scatter\", mode = \"lines\",",
  "      line = list(",
  "        color = unname(model_colors[lv]),",
  "        width = if (lv == \"Actual\") 2.2 else 1.4,",
  "        dash  = if (lv == \"Actual\") \"solid\" else \"dot\"",
  "      ),",
  "      hovertemplate = paste0(",
  "        \"<b>\", lv,",
  "        \"</b><br>%{x|%Y-%m-%d}: %{y:.1f} µg/m³<extra></extra>\"",
  "      )",
  "    )",
  "  }",
  "  p %>%",
  "    plotly::layout(",
  "      autosize = TRUE,",
  "      title = list(text = paste0(\"<b>\", cn, \"</b>\"), x = 0.02, y = 0.97),",
  "      xaxis = list(title = NULL, rangeslider = list(visible = TRUE, thickness = 0.08)),",
  "      yaxis = list(title = \"PM₂.₅ (µg/m³)\"),",
  "      hovermode = \"x unified\",",
  "      margin = list(t = 80, b = 60, l = 60, r = 20),",
  "      legend = list(",
  "        orientation = \"h\",",
  "        x = 0, xanchor = \"left\",",
  "        y = 1.08, yanchor = \"bottom\",",
  "        bgcolor = \"rgba(255,255,255,0.85)\",",
  "        bordercolor = \"#ccc\", borderwidth = 1,",
  "        font = list(size = 11)",
  "      ),",
  "      shapes = list(list(",
  "        type = \"line\",",
  "        x0 = min(city_df$date), x1 = max(city_df$date),",
  "        y0 = 15, y1 = 15,",
  "        line = list(color = \"red\", width = 1, dash = \"dash\")",
  "      )),",
  "      annotations = list(list(",
  "        x = min(city_df$date), y = 15, xref = \"x\", yref = \"y\",",
  "        text = \"WHO 2021 AQG 15 µg/m³\", showarrow = FALSE,",
  "        xanchor = \"left\", yanchor = \"bottom\",",
  "        font = list(color = \"red\", size = 10)",
  "      ))",
  "    ) %>%",
  "    plotly::config(responsive = TRUE)",
  "}",
  "",
  "processed_dir <- here::here(\"data\", \"processed\")",
  "if (!dir.exists(processed_dir)) dir.create(processed_dir, recursive = TRUE)",
  "build_fn_path <- file.path(processed_dir, \"city_forecasts_build_fn.R\")",
  "writeLines(",
  "  c(\"build_city_plot <- \", deparse(build_city_plot)),",
  "  build_fn_path",
  ")",
  "",
  "docs_dir <- here::here(\"docs\")",
  "if (!dir.exists(docs_dir)) dir.create(docs_dir, recursive = TRUE)",
  "# Render from a project-local qmd (not a tempfile): Quarto resolves resource",
  "# paths relative to the qmd's location, and a tempfile in /var/folders forces",
  "# pandoc to climb to the filesystem root and fail with a permission error.",
  "qmd_tmp <- file.path(docs_dir, \"_city_forecasts.qmd\")",
  "",
  "header <- c(",
  "  \"---\",",
  "  \"title: \\\"PM2.5 forecasts — top Polish cities\\\"\",",
  "  sprintf(",
  "    \"subtitle: \\\"Actual vs. predicted at the nearest 10 km cell across the %d-day test window\\\"\",",
  "    length(test_dates)",
  "  ),",
  "  \"format:\",",
  "  \"  html:\",",
  "  \"    embed-resources: true\",",
  "  \"    toc: true\",",
  "  \"    toc-depth: 2\",",
  "  \"    toc-location: left\",",
  "  \"    theme: cosmo\",",
  "  \"    page-layout: full\",",
  "  \"    grid:\",",
  "  \"      body-width: 1200px\",",
  "  \"      sidebar-width: 240px\",",
  "  \"      margin-width: 100px\",",
  "  \"---\",",
  "  \"\",",
  "  \"```{r setup, include=FALSE}\",",
  "  \"knitr::opts_chunk$set(echo = FALSE, warning = FALSE, message = FALSE)\",",
  "  \"library(plotly); library(dplyr); library(knitr)\",",
  "  # Absolute paths baked in: the qmd renders from a temp/other dir where here::here()",
  "  # cannot find the project root, so readRDS(here::here(...)) would fail to open.",
  "  sprintf(\"city_series  <- readRDS(%s)\",",
  "          encodeString(file.path(processed_dir, \"city_forecasts_series.rds\"), quote = \"\\\"\")),",
  "  sprintf(\"city_metrics <- readRDS(%s)\",",
  "          encodeString(file.path(processed_dir, \"city_forecasts_metrics.rds\"), quote = \"\\\"\")),",
  "  sprintf(\"model_colors <- readRDS(%s)\",",
  "          encodeString(file.path(processed_dir, \"city_forecasts_model_colors.rds\"), quote = \"\\\"\")),",
  "  sprintf(\"source(%s)\", encodeString(build_fn_path, quote = \"\\\"\")),",
  "  \"```\",",
  "  \"\",",
  "  \"## About this dashboard {.unnumbered}\",",
  "  \"\",",
  "  sprintf(",
  "    \"Each chart below shows the daily actual PM2.5 (solid black line) and every trained model's one-day-ahead forecast (dotted, colour-coded) at the grid cell closest to a given city, across the %d-day held-out test window. The red dashed line marks the WHO 2021 Global Air Quality Guideline 24-hour PM2.5 limit of 15 µg/m³. Use the range slider, hover on a date for exact values, or click the legend to toggle individual traces.\",",
  "    length(test_dates)",
  "  ),",
  "  \"\"",
  ")",
  "",
  "city_chunks <- unlist(lapply(cities_df$name, function(cn) {",
  "  c(",
  "    sprintf(\"## %s {#%s}\", cn, gsub(\"[^A-Za-z0-9]+\", \"-\", cn)),",
  "    \"\",",
  "    sprintf(\"```{r fig-%s}\", gsub(\"[^A-Za-z0-9]+\", \"-\", cn)),",
  "    \"#| column: page\",",
  "    \"#| out-width: 100%\",",
  "    sprintf(\"build_city_plot(city_series[[\\\"%s\\\"]], \\\"%s\\\")\", cn, cn),",
  "    \"```\",",
  "    \"\",",
  "    sprintf(\"```{r tbl-%s}\", gsub(\"[^A-Za-z0-9]+\", \"-\", cn)),",
  "    sprintf(\"city_metrics %%>%% filter(City == \\\"%s\\\") %%>%% arrange(RMSE) %%>%% select(-City) %%>%% kable(digits = 2, caption = \\\"Per-city test-set accuracy (sorted by RMSE)\\\")\", cn),",
  "    \"```\",",
  "    \"\"",
  "  )",
  "}))",
  "",
  "writeLines(c(header, city_chunks), qmd_tmp)",
  "",
  "processed_dir <- here::here(\"data\", \"processed\")",
  "saveRDS(city_series, file.path(processed_dir, \"city_forecasts_series.rds\"))",
  "saveRDS(city_metrics, file.path(processed_dir, \"city_forecasts_metrics.rds\"))",
  "saveRDS(model_colors, file.path(processed_dir, \"city_forecasts_model_colors.rds\"))",
  "",
  "out_html   <- here::here(\"docs\", \"city_forecasts.html\")",
  "quarto_bin <- Sys.which(\"quarto\")",
  "qmd_base   <- basename(qmd_tmp)",
  "files_dir  <- file.path(docs_dir, paste0(tools::file_path_sans_ext(qmd_base), \"_files\"))",
  "old_wd     <- getwd()",
  "tryCatch({",
  "  # Render in docs_dir so output and resource paths stay inside the project.",
  "  setwd(docs_dir)",
  "  if (requireNamespace(\"quarto\", quietly = TRUE)) {",
  "    quarto::quarto_render(qmd_base, output_format = \"html\",",
  "                          output_file = basename(out_html))",
  "  } else if (nzchar(quarto_bin)) {",
  "    system2(quarto_bin,",
  "      args = c(\"render\", qmd_base, \"--to\", \"html\", \"--output\", basename(out_html)))",
  "  } else {",
  "    rmarkdown::render(qmd_base, output_file = basename(out_html),",
  "                      output_dir = \".\", quiet = FALSE)",
  "  }",
  "}, error = function(e) {",
  "  warning(\"city-forecast dashboard render skipped (no working quarto/rmarkdown): \",",
  "          conditionMessage(e), call. = FALSE)",
  "}, finally = {",
  "  setwd(old_wd)",
  "  unlink(c(qmd_tmp, files_dir), recursive = TRUE)",
  "})",
  "",
  "if (file.exists(out_html)) {",
  "  cat(sprintf(\"\\nInteractive dashboard saved to: %s\\n\", out_html))",
  "} else {",
  "  cat(\"\\nDashboard not rendered (no working quarto/rmarkdown); data sidecars saved under data/processed/.\\n\")",
  "}",
  "cat(sprintf(",
  "  \"  Cities: %d | Models: %d | Test days: %d\\n\",",
  "  nrow(cities_df), length(model_preds), length(test_dates)",
  "))"
))
files_written <- c(files_written, "code/11_city_forecasts.R")

write_code_file("code/12_didactic_figures.R", c(
  "suppressPackageStartupMessages({",
  "  library(tidyverse)",
  "  library(here)",
  "  library(patchwork)",
  "})",
  "",
  "fig_dir <- here::here(\"output\", \"figures\")",
  "dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)",
  "",
  "theme_didactic <- ggplot2::theme_void(base_size = 12) +",
  "  ggplot2::theme(",
  "    plot.title       = ggplot2::element_text(hjust = 0.5, size = 13, face = \"bold\"),",
  "    plot.subtitle    = ggplot2::element_text(hjust = 0.5, size = 10, colour = \"grey30\"),",
  "    plot.margin      = ggplot2::margin(8, 8, 8, 8),",
  "    legend.position  = \"none\"",
  "  )",
  "",
  "box_node <- function(xmin, xmax, ymin, ymax, fill, label) {",
  "  list(",
  "    ggplot2::annotate(\"rect\", xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax,",
  "             fill = fill, colour = \"grey20\", linewidth = 0.4),",
  "    ggplot2::annotate(\"text\", x = (xmin + xmax) / 2, y = (ymin + ymax) / 2,",
  "             label = label, size = 4.4, parse = TRUE)",
  "  )",
  "}",
  "",
  "op_circle <- function(x, y, label, fill = \"white\", label_size = 5.2) {",
  "  list(",
  "    ggplot2::annotate(\"point\", x = x, y = y, shape = 21, fill = fill,",
  "             colour = \"grey20\", stroke = 0.6, size = 11),",
  "    ggplot2::annotate(\"text\", x = x, y = y, label = label, size = label_size)",
  "  )",
  "}",
  "",
  "arrow_seg <- function(x, xend, y, yend, lty = \"solid\", lwd = 0.55) {",
  "  ggplot2::annotate(\"segment\", x = x, xend = xend, y = y, yend = yend,",
  "           arrow = grid::arrow(length = grid::unit(0.10, \"cm\"), type = \"closed\"),",
  "           linewidth = lwd, linetype = lty, colour = \"grey20\")",
  "}",
  "",
  "plain_seg <- function(x, xend, y, yend, lty = \"solid\", lwd = 0.55) {",
  "  ggplot2::annotate(\"segment\", x = x, xend = xend, y = y, yend = yend,",
  "           linewidth = lwd, linetype = lty, colour = \"grey20\")",
  "}",
  "",
  "col_state <- \"#FCE4A6\"",
  "col_gate  <- \"#CFE8FF\"",
  "col_cand  <- \"#D9F0D3\"",
  "col_op    <- \"white\"",
  "",
  "y_top <- 5.5",
  "y_bot <- 1.3",
  "",
  "gx <- c(f = 2.7, i = 4.7, g = 6.7, o = 8.7)",
  "gw <- 0.85",
  "",
  "mul_f     <- gx[\"f\"]",
  "mul_ig    <- (gx[\"i\"] + gx[\"g\"]) / 2",
  "plus_node <- mul_ig",
  "mul_o     <- 10.3",
  "tanh_node <- 10.3",
  "",
  "p_lstm <- ggplot2::ggplot() +",
  "  ggplot2::annotate(\"rect\", xmin = 0.5, xmax = 11.0, ymin = 0.4, ymax = 6.4,",
  "           fill = \"grey97\", colour = \"grey60\", linewidth = 0.3,",
  "           linetype = \"dashed\") +",
  "  ggplot2::annotate(\"text\", x = 10.9, y = 6.25, label = \"LSTM cell\",",
  "           hjust = 1, vjust = 1, size = 3.2,",
  "           colour = \"grey45\", fontface = \"italic\") +",
  "",
  "  plain_seg(0.6, mul_f - 0.18, y_top, y_top, lwd = 1.0) +",
  "  op_circle(mul_f, y_top, \"×\", fill = col_op) +",
  "  plain_seg(mul_f + 0.18, plus_node - 0.18, y_top, y_top, lwd = 1.0) +",
  "  op_circle(plus_node, y_top, \"+\", fill = col_op) +",
  "  plain_seg(plus_node + 0.18, 10.9, y_top, y_top, lwd = 1.0) +",
  "  ggplot2::annotate(\"text\", x = 0.55, y = y_top + 0.35, label = \"c[t-1]\",",
  "           parse = TRUE, hjust = 0, size = 4.4) +",
  "  ggplot2::annotate(\"text\", x = 10.95, y = y_top + 0.35, label = \"c[t]\",",
  "           parse = TRUE, hjust = 1, size = 4.4) +",
  "  ggplot2::annotate(\"text\", x = (mul_f + plus_node) / 2, y = y_top + 0.55,",
  "           label = \"additive update (CEC)\",",
  "           size = 3.0, colour = \"#B8860B\", fontface = \"italic\") +",
  "",
  "  plain_seg(0.6, mul_o + 0.18, y_bot, y_bot, lwd = 0.5, lty = \"dotted\") +",
  "  plain_seg(mul_o, 10.9, y_bot, y_bot, lwd = 1.0) +",
  "  ggplot2::annotate(\"text\", x = 0.55, y = y_bot + 0.35, label = \"h[t-1]\",",
  "           parse = TRUE, hjust = 0, size = 4.4) +",
  "  ggplot2::annotate(\"text\", x = 10.95, y = y_bot + 0.35, label = \"h[t]\",",
  "           parse = TRUE, hjust = 1, size = 4.4) +",
  "",
  "  plain_seg(0.6, gx[\"o\"] + 0.0, 0.7, 0.7, lwd = 0.5, lty = \"dotted\") +",
  "  ggplot2::annotate(\"text\", x = 0.55, y = 0.7, label = \"x[t]\",",
  "           parse = TRUE, hjust = 1.1, size = 4.4) +",
  "",
  "  box_node(gx[\"f\"] - gw, gx[\"f\"] + gw, 2.5, 3.5, col_gate, \"f[t]\") +",
  "  box_node(gx[\"i\"] - gw, gx[\"i\"] + gw, 2.5, 3.5, col_gate, \"i[t]\") +",
  "  box_node(gx[\"g\"] - gw, gx[\"g\"] + gw, 2.5, 3.5, col_cand, \"g[t]\") +",
  "  box_node(gx[\"o\"] - gw, gx[\"o\"] + gw, 2.5, 3.5, col_gate, \"o[t]\") +",
  "",
  "  arrow_seg(gx[\"f\"], gx[\"f\"], y_bot, 2.5) +",
  "  arrow_seg(gx[\"i\"], gx[\"i\"], y_bot, 2.5) +",
  "  arrow_seg(gx[\"g\"], gx[\"g\"], y_bot, 2.5) +",
  "  arrow_seg(gx[\"o\"], gx[\"o\"], y_bot, 2.5) +",
  "",
  "  arrow_seg(gx[\"f\"], gx[\"f\"], 3.5, y_top - 0.18) +",
  "  plain_seg(gx[\"i\"], gx[\"i\"], 3.5, 4.4, lwd = 0.5) +",
  "  plain_seg(gx[\"g\"], gx[\"g\"], 3.5, 4.4, lwd = 0.5) +",
  "  plain_seg(gx[\"i\"], mul_ig - 0.16, 4.4, 4.4, lwd = 0.5) +",
  "  plain_seg(mul_ig + 0.16, gx[\"g\"], 4.4, 4.4, lwd = 0.5) +",
  "  op_circle(mul_ig, 4.4, \"×\", fill = col_op, label_size = 4.4) +",
  "  arrow_seg(mul_ig, plus_node, 4.4, y_top - 0.18) +",
  "",
  "  op_circle(10.3, 4.4, \"tanh\", fill = col_state, label_size = 3.4) +",
  "  plain_seg(10.3, 10.3, y_top - 0.18, 4.6, lwd = 0.5) +",
  "  plain_seg(10.3, 10.3, 4.2, 3.6, lwd = 0.5) +",
  "  op_circle(10.3, 3.4, \"×\", fill = col_op, label_size = 4.4) +",
  "  plain_seg(gx[\"o\"] + gw, gx[\"o\"] + gw, 3.0, 3.4, lwd = 0.5) +",
  "  plain_seg(gx[\"o\"] + gw, 10.15, 3.4, 3.4, lwd = 0.5) +",
  "  arrow_seg(10.3, 10.3, 3.22, y_bot + 0.05) +",
  "",
  "  ggplot2::coord_fixed(xlim = c(-0.2, 11.4), ylim = c(0.0, 6.6), expand = FALSE) +",
  "  ggplot2::labs(",
  "    title    = \"The LSTM cell: an additive cell-state highway with multiplicative gates\",",
  "    subtitle = expression(\"Cell state \" * c[t] == f[t] %.% c[t-1] + i[t] %.% g[t] *",
  "                          \".  The '+' on the highway lets gradients flow back through time unimpeded.\")",
  "  ) +",
  "  theme_didactic",
  "",
  "gate_eqs_top <- expression(",
  "  paste(f[t] == sigma(W[\"xf\"] * x[t] + W[\"hf\"] * h[t-1] + b[f]),",
  "        \"          \",",
  "        i[t] == sigma(W[\"xi\"] * x[t] + W[\"hi\"] * h[t-1] + b[i]))",
  ")",
  "gate_eqs_bot <- expression(",
  "  paste(g[t] == tanh(W[\"xg\"] * x[t] + W[\"hg\"] * h[t-1] + b[g]),",
  "        \"          \",",
  "        o[t] == sigma(W[\"xo\"] * x[t] + W[\"ho\"] * h[t-1] + b[o]))",
  ")",
  "",
  "caption_panel <- ggplot2::ggplot() +",
  "  ggplot2::annotate(\"text\", x = 0, y = 0.85, label = gate_eqs_top,",
  "           size = 3.4, hjust = 0.5, parse = TRUE) +",
  "  ggplot2::annotate(\"text\", x = 0, y = 0.50, label = gate_eqs_bot,",
  "           size = 3.4, hjust = 0.5, parse = TRUE) +",
  "  ggplot2::annotate(\"text\", x = 0, y = 0.0,",
  "           label = expression(paste(c[t] == f[t] %.% c[t-1] + i[t] %.% g[t],",
  "                                    \"        \",",
  "                                    h[t] == o[t] %.% tanh(c[t]))),",
  "           size = 3.8, hjust = 0.5, parse = TRUE, fontface = \"italic\") +",
  "  ggplot2::coord_cartesian(xlim = c(-1, 1), ylim = c(-0.3, 1.1)) +",
  "  ggplot2::theme_void()",
  "",
  "p_lstm_full <- p_lstm / caption_panel + patchwork::plot_layout(heights = c(7, 1.5))",
  "",
  "ggsave_quiet <- function(...) {",
  "  withCallingHandlers(",
  "    ggplot2::ggsave(...),",
  "    warning = function(w) {",
  "      if (grepl(\"applied to non-.*'expression'\", conditionMessage(w))) invokeRestart(\"muffleWarning\")",
  "    }",
  "  )",
  "}",
  "",
  "ggsave_quiet(file.path(fig_dir, \"lstm_cell_diagram.png\"),",
  "       p_lstm_full, width = 11, height = 6.4, dpi = 200, bg = \"white\")",
  "",
  "set.seed(7)",
  "input_mat  <- matrix(c(",
  "  3, 1, 0, 2, 1,",
  "  2, 4, 1, 0, 2,",
  "  1, 2, 3, 1, 0,",
  "  0, 1, 2, 3, 1,",
  "  2, 0, 1, 2, 3",
  "), nrow = 5, byrow = TRUE)",
  "",
  "kernel_mat <- matrix(c(",
  "  1,  0, -1,",
  "  1,  0, -1,",
  "  1,  0, -1",
  "), nrow = 3, byrow = TRUE)",
  "",
  "out_row1 <- sapply(1:3, function(s) sum(input_mat[1:3, s:(s + 2)] * kernel_mat))",
  "",
  "grid_to_df <- function(M, x_off = 0, y_off = 0) {",
  "  nr <- nrow(M); nc <- ncol(M)",
  "  tidyr::expand_grid(row = seq_len(nr), col = seq_len(nc)) |>",
  "    dplyr::mutate(",
  "      x     = col + x_off,",
  "      y     = (nr - row + 1) + y_off,",
  "      value = as.vector(t(M))",
  "    )",
  "}",
  "",
  "INP_X <- 0",
  "KER_X <- 6",
  "OUT_X <- 10",
  "Y_OFF <- 1",
  "",
  "conv_panel <- function(slide) {",
  "  patch <- input_mat[1:3, slide:(slide + 2)]",
  "  out_val <- out_row1[slide]",
  "",
  "  inp_df <- grid_to_df(input_mat, x_off = INP_X) |>",
  "    dplyr::mutate(highlight = (col >= slide & col <= slide + 2 & row <= 3))",
  "",
  "  ker_df <- grid_to_df(kernel_mat, x_off = KER_X, y_off = Y_OFF)",
  "",
  "  out_grid <- matrix(NA_real_, nrow = 3, ncol = 3)",
  "  for (s in seq_len(slide)) out_grid[1, s] <- out_row1[s]",
  "  out_df <- grid_to_df(out_grid, x_off = OUT_X, y_off = Y_OFF) |>",
  "    dplyr::filter(!is.na(value)) |>",
  "    dplyr::mutate(is_current = col == slide)",
  "",
  "  out_frame <- tidyr::expand_grid(row = 1:3, col = 1:3) |>",
  "    dplyr::mutate(x = col + OUT_X, y = (3 - row + 1) + Y_OFF)",
  "",
  "  prods <- sprintf(\"%d·%d\",",
  "                   as.vector(t(kernel_mat)),",
  "                   as.vector(t(patch)))",
  "  eq_rhs <- paste(prods, collapse = \" + \")",
  "",
  "  ggplot2::ggplot() +",
  "    ggplot2::geom_tile(data = inp_df, ggplot2::aes(x = x, y = y, fill = highlight),",
  "              colour = \"grey30\", linewidth = 0.3) +",
  "    ggplot2::geom_text(data = inp_df, ggplot2::aes(x = x, y = y, label = value),",
  "              size = 3.6) +",
  "    ggplot2::scale_fill_manual(values = c(`TRUE` = \"#FFE9B0\", `FALSE` = \"white\")) +",
  "    ggplot2::annotate(\"text\", x = INP_X + 3, y = 6.6, label = \"input  U  (5×5)\",",
  "             size = 3.6) +",
  "",
  "    ggplot2::geom_tile(data = ker_df, ggplot2::aes(x = x, y = y),",
  "              fill = \"#CFE8FF\", colour = \"grey30\", linewidth = 0.3) +",
  "    ggplot2::geom_text(data = ker_df, ggplot2::aes(x = x, y = y, label = value),",
  "              size = 3.6) +",
  "    ggplot2::annotate(\"text\", x = KER_X + 2, y = 6.6, label = \"kernel  K  (3×3)\",",
  "             size = 3.6) +",
  "",
  "    ggplot2::geom_tile(data = out_frame, ggplot2::aes(x = x, y = y),",
  "              fill = \"white\", colour = \"grey80\", linewidth = 0.3) +",
  "    ggplot2::geom_tile(data = out_df,",
  "              ggplot2::aes(x = x, y = y,",
  "                  fill = is_current),",
  "              colour = \"grey30\", linewidth = 0.4, show.legend = FALSE) +",
  "    ggplot2::geom_text(data = out_df, ggplot2::aes(x = x, y = y, label = value),",
  "              size = 4.2, fontface = \"bold\") +",
  "    ggplot2::annotate(\"text\", x = OUT_X + 2, y = 6.6, label = \"output  V  (3×3)\",",
  "             size = 3.6) +",
  "",
  "    ggplot2::annotate(\"text\", x = (INP_X + OUT_X + 3) / 2, y = -0.2,",
  "             label = sprintf(\"V[1,%d] = %s = %d\", slide, eq_rhs, out_val),",
  "             size = 3.1, colour = \"grey15\") +",
  "",
  "    ggplot2::coord_fixed(xlim = c(-0.4, OUT_X + 4.5), ylim = c(-0.7, 7),",
  "                expand = FALSE) +",
  "    theme_didactic",
  "}",
  "",
  "p_conv <- (conv_panel(1) / conv_panel(2) / conv_panel(3)) +",
  "  patchwork::plot_annotation(",
  "    title    = \"A 3×3 kernel walks across a 5×5 input\",",
  "    subtitle = \"The same nine kernel weights are applied at every position; each output cell is the sum of the elementwise product\",",
  "    theme    = ggplot2::theme(",
  "      plot.title    = ggplot2::element_text(hjust = 0.5, size = 13, face = \"bold\"),",
  "      plot.subtitle = ggplot2::element_text(hjust = 0.5, size = 10, colour = \"grey30\")",
  "    )",
  "  )",
  "",
  "ggsave_quiet(file.path(fig_dir, \"conv2d_in_action.png\"),",
  "       p_conv, width = 11, height = 9.2, dpi = 200, bg = \"white\")",
  "",
  "rf_panel <- function(L, rf_size, grid_size = 15) {",
  "  centre <- (grid_size + 1) / 2",
  "  half   <- (rf_size - 1) / 2",
  "  df <- tidyr::expand_grid(row = seq_len(grid_size), col = seq_len(grid_size)) |>",
  "    dplyr::mutate(",
  "      in_rf  = abs(row - centre) <= half & abs(col - centre) <= half,",
  "      is_ctr = row == centre & col == centre",
  "    )",
  "",
  "  ggplot2::ggplot(df, ggplot2::aes(x = col, y = grid_size - row + 1)) +",
  "    ggplot2::geom_tile(ggplot2::aes(fill = in_rf), colour = \"grey80\", linewidth = 0.2) +",
  "    ggplot2::geom_tile(data = dplyr::filter(df, is_ctr), fill = \"#E63946\",",
  "              colour = \"grey20\", linewidth = 0.4) +",
  "    ggplot2::scale_fill_manual(values = c(`TRUE` = \"#FFD27A\", `FALSE` = \"grey97\")) +",
  "    ggplot2::coord_fixed(expand = FALSE) +",
  "    ggplot2::labs(",
  "      title    = sprintf(\"After %d stride-2 conv layer%s\", L, if (L == 1) \"\" else \"s\"),",
  "      subtitle = sprintf(\"receptive field ≈ %d×%d input cells\",",
  "                         rf_size, rf_size)",
  "    ) +",
  "    theme_didactic +",
  "    ggplot2::theme(",
  "      plot.title    = ggplot2::element_text(hjust = 0.5, size = 11, face = \"bold\"),",
  "      plot.subtitle = ggplot2::element_text(hjust = 0.5, size = 9.5, colour = \"grey30\")",
  "    )",
  "}",
  "",
  "p_rf <- rf_panel(1, 3) + rf_panel(2, 7) + rf_panel(3, 15) +",
  "  patchwork::plot_annotation(",
  "    title    = \"Stacking convolutions widens the receptive field\",",
  "    subtitle = \"Red = the output cell. Yellow = the input cells it depends on. Each stride-2 layer roughly doubles the field (3 → 7 → 15 cells).\",",
  "    theme    = ggplot2::theme(",
  "      plot.title    = ggplot2::element_text(hjust = 0.5, size = 13, face = \"bold\"),",
  "      plot.subtitle = ggplot2::element_text(hjust = 0.5, size = 10, colour = \"grey30\")",
  "    )",
  "  )",
  "",
  "ggsave_quiet(file.path(fig_dir, \"receptive_field_growth.png\"),",
  "       p_rf, width = 10, height = 4.4, dpi = 200, bg = \"white\")",
  "",
  "message(\"Wrote three didactic figures to \", fig_dir)"
))
files_written <- c(files_written, "code/12_didactic_figures.R")

write_code_file("code/run_all_after_data_acquisition.R", c(
  "# run_all_after_data_acquisition.R",
  "# Runs the full pipeline in order, starting AFTER data acquisition — i.e.",
  "# everything from 02_data_prep.R through 12_didactic_figures.R. Assumes",
  "# 00_install.R (packages) and 01_data_acquisition.R (raw download) have",
  "# already been run; this script does NOT re-fetch the raw data.",
  "#",
  "# The six model scripts (04-09) each call load_or_train(), so they luz_load",
  "# their checkpoint from models/ when present and only train fresh when the",
  "# corresponding .pt is absent (see code/_common_training.R). To force a",
  "# retrain, delete the relevant models/<name>_final.pt before running.",
  "#",
  "# Usage:",
  "#   Rscript code/run_all_after_data_acquisition.R",
  "#",
  "# Retraining honours the TRAINING_PROFILE env var (see _common_training.R):",
  "#   TRAINING_PROFILE=gpu Rscript code/run_all_after_data_acquisition.R",
  "# trains the CNN-LSTM/ConvLSTM at batch 64 / lr 4e-3 on CUDA when available;",
  "# the default \"cpu\" profile uses batch 4 / lr 1e-3 everywhere.",
  "#",
  "# Wall-clock: ~10-15 min on a laptop CPU for scripts 03-12 with all 6",
  "# checkpoints present (CPU predict only); +20-30 min for each model whose",
  "# checkpoint is absent and so must train fresh. 02_data_prep.R adds a few",
  "# minutes on top of that.",
  "#",
  "# Each script runs in the GLOBAL environment and the objects it created are",
  "# removed afterwards (then gc), so peak RAM still tracks the single heaviest",
  "# script instead of accumulating the ~1.35 GB source_array across all eleven.",
  "#",
  "# globalenv() (not a throwaway new.env()) is REQUIRED: the model scripts",
  "# checkpoint via luz_save(), which serialises the fitted torch module, and",
  "# R serialize() copies non-blessed environments BY VALUE. In a new.env() the",
  "# module closure reaches the env holding source_array (referenced again by",
  "# every dataloader dataset), so luz_save() writes the array several times and",
  "# overflows R 2 GB long-vector limit (\"long vectors not supported yet\").",
  "# globalenv() is blessed and serialised by reference, so the array is never",
  "# copied into the checkpoint (as in the legacy notebook, whose cells = globalenv).",
  "",
  "library(here)",
  "",
  "scripts <- c(",
  "  \"02_data_prep.R\",",
  "  \"03_baselines.R\",",
  "  \"04_cnn_lstm_skip.R\",",
  "  \"05_cnn_lstm_noskip.R\",",
  "  \"06_lstm_only_skip.R\",",
  "  \"07_lstm_only_noskip.R\",",
  "  \"08_convlstm_skip.R\",",
  "  \"09_convlstm_noskip.R\",",
  "  \"10_evaluation.R\",",
  "  \"11_city_forecasts.R\",",
  "  \"12_didactic_figures.R\"",
  ")",
  "",
  ".runner_keep <- c(ls(globalenv(), all.names = TRUE), \".runner_keep\", \"s\", \"t0\")",
  "",
  "for (s in scripts) {",
  "  cat(sprintf(\"\\n\\n========== %s ==========\\n\\n\", s))",
  "  t0 <- Sys.time()",
  "  source(here::here(\"code\", s), local = globalenv(), echo = FALSE,",
  "         max.deparse.length = 500)",
  "  rm(list = setdiff(ls(globalenv(), all.names = TRUE), .runner_keep),",
  "     envir = globalenv())",
  "  invisible(gc())",
  "  cat(sprintf(\"\\n[%s finished in %.1f min]\\n\",",
  "              s, as.numeric(difftime(Sys.time(), t0, units = \"mins\"))))",
  "}",
  "",
  "cat(\"\\n\\nDone. Pipeline (02-12) finished.\\n\")"
))
files_written <- c(files_written, "code/run_all_after_data_acquisition.R")

cat(sprintf("materialized %d project files\n", length(files_written)))


## 5. Install R packages and LibTorch


In [ ]:
say("run 00_install.R")
source("code/00_install.R")

cuda_ok <- isTRUE(try(torch::cuda_is_available(), silent = TRUE))
cat("torch CUDA available:", cuda_ok, "\n")


## 5b. Verify the CUDA build for `torch`

The current `00_install.R` installs the standard `torch` build. On Kaggle, this cell tries to switch to a CUDA-enabled build if LibTorch does not yet see the GPU.


In [ ]:
say("ensure CUDA-enabled LibTorch")
cuda_ok <- isTRUE(try(torch::cuda_is_available(), silent = TRUE))
if (!cuda_ok) {
  for (build in c("cu121", "cu118")) {
    cat(sprintf("Trying torch CUDA build: %s\n", build))
    try(torch::install_torch(type = build), silent = TRUE)
    cuda_ok <- isTRUE(try(torch::cuda_is_available(), silent = TRUE))
    if (cuda_ok) break
  }
}
cat("torch CUDA available after Kaggle check:", cuda_ok, "\n")
if (!cuda_ok) message("torch still sees no CUDA; the notebook will still run, but training may fall back to CPU.")


## 6. Download input data


In [ ]:
say("run 01_data_acquisition.R")
if (!nzchar(CDS_USER) || !nzchar(CDS_API_KEY)) {
  stop("Set CDS_USER and CDS_API_KEY in the configuration cell before running this step.")
}
source("code/01_data_acquisition.R")


## 7. Verify CAMS completeness


In [ ]:
expected <- sprintf("cams_pm25_poland_%d_%02d", rep(2018:2022, each = 12), rep(1:12, times = 5))
have_nc  <- file.exists(file.path("data/raw", paste0(expected, ".nc")))
have_zip <- file.exists(file.path("data/raw", paste0(expected, ".zip")))
present  <- have_nc | have_zip
cat(sprintf("CAMS months present: %d / %d\n", sum(present), length(present)))
stopifnot("CAMS download incomplete" = all(present))


## 8. Run pipeline 02-12

The notebook uses the current `run_all_after_data_acquisition.R`, so execution order and control flow stay aligned with the current project.


In [ ]:
say("run pipeline 02-12")
source("code/run_all_after_data_acquisition.R")


## 9. Collect downloadable artifacts


In [ ]:
say("collect artifacts")
if (dir.exists("data/raw")) unlink("data/raw", recursive = TRUE)
tables <- list.files("output/tables", full.names = TRUE)
figures <- list.files("output/figures", pattern = "\\.png$", full.names = TRUE)
models <- list.files("models", pattern = "\\.pt$", full.names = TRUE)
docs_files <- c(if (file.exists("docs/city_forecasts.html")) "docs/city_forecasts.html")
processed <- c(
  if (file.exists("data/processed/pm25_tensors.rds")) "data/processed/pm25_tensors.rds",
  if (file.exists("data/processed/spatial_features.rds")) "data/processed/spatial_features.rds",
  list.files("data/processed", pattern = "_test_results\\.rds$", full.names = TRUE)
)
bundle <- unique(c(tables, figures, models, docs_files, processed))
bundle <- bundle[file.exists(bundle)]
cat(sprintf("tables: %d | figures: %d | models: %d | bundled files: %d\n",
            length(tables), length(figures), length(models), length(bundle)))
if (file.exists("kaggle_outputs.zip")) unlink("kaggle_outputs.zip")
if (length(bundle) > 0) utils::zip(zipfile = "kaggle_outputs.zip", files = bundle)
cat("Saved outputs under /kaggle/working and kaggle_outputs.zip\n")
